# PROJET ANALYSE DES DONNÉES
## Axe Finance - Apple (AAPL)
## ÉTAPE 2 : ANALYSE UNIVARIÉE

### Objectifs :
1. Calculer les mesures statistiques pour chaque variable
2. Analyser les distributions
3. Gérer les valeurs manquantes et aberrantes
4. Visualiser chaque variable avec des graphiques appropriés

In [ ]:
# ======================
# IMPORTATION DES LIBRAIRIES
# ======================

print(" Installation et importation des packages...")

# Installation si nécessaire
try:
    import yfinance as yf
    print(" yfinance déjà installé")
except ImportError:
    print(" Installation de yfinance...")
    !pip install yfinance -q
    import yfinance as yf

# Importation standard
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print(" Toutes les librairies sont importées !")

In [ ]:
# ======================
# TÉLÉCHARGEMENT DES DONNÉES
# ======================

print(" TÉLÉCHARGEMENT DES DONNÉES APPLE (AAPL)")
print("=" * 50)

# Paramètres
TICKER = "AAPL"
DATE_DEBUT = "2020-01-01"
DATE_FIN = "2024-12-01"

print(f" Actif analysé : {TICKER}")
print(f" Période : {DATE_DEBUT} au {DATE_FIN}")

# Téléchargement
data = yf.download(TICKER, start=DATE_DEBUT, end=DATE_FIN, progress=False)

# Vérification
print(f"    Téléchargement réussi !")
print(f"   • Nombre de jours : {len(data)}")
print(f"   • Colonnes : {list(data.columns)}")
print(f"   • Période : {data.index[0].date()} au {data.index[-1].date()}")

# Aperçu
print("\n APERÇU DES DONNÉES :")
data.head()

In [ ]:
# ========================================
# PRÉPARATION DES DONNÉES POUR L'ANALYSE
# ========================================

print("🔧 PRÉPARATION DES DONNÉES")
print("=" * 50)

# Créer une copie propre
df = data.copy()

# 1. Création des variables dérivées
df['Rendement_%'] = df['Close'].pct_change() * 100  # Rendements en %
df['Rendement_Log'] = np.log(df['Close'] / df['Close'].shift(1)) * 100  # Rendements logarithmiques
df['Volume_Log'] = np.log(df['Volume'] + 1)  # Transformation log du volume

# 2. Supprimer la première ligne (NaN due au pct_change)
df_clean = df.dropna().copy()

print(f" Données préparées")
print(f"   • Lignes initiales : {len(df)}")
print(f"   • Lignes après nettoyage : {len(df_clean)}")
print(f"   • Colonnes disponibles : {list(df_clean.columns)}")

# Aperçu
df_clean.head()

In [ ]:
# ========================================
# STATISTIQUES DESCRIPTIVES 
# ========================================

print("📈 STATISTIQUES DESCRIPTIVES - ANALYSE UNIVARIÉE")
print("=" * 60)

# --------------------------------------------------
# 1) Sécurisation : garantir des colonnes simples
# --------------------------------------------------
if isinstance(df_clean.columns, pd.MultiIndex):
    df_clean.columns = [col[0] for col in df_clean.columns]

# --------------------------------------------------
# 2) Fonction d'analyse univariée robuste
# --------------------------------------------------
def analyse_variable_corrigee(nom, serie):
    print(f"\n📊 {nom.upper()} :")
    
    # Forcer une Series 1D
    if isinstance(serie, pd.DataFrame):
        serie = serie.iloc[:, 0]
    
    # Conversion numérique sécurisée
    serie = pd.to_numeric(serie, errors='coerce')
    serie_clean = serie.dropna()
    
    if serie_clean.empty:
        print("   ❌ Pas de données valides")
        return
    
    # Statistiques descriptives
    print(f"   • Observations     : {len(serie_clean)}")
    print(f"   • Moyenne          : {serie_clean.mean():.4f}")
    print(f"   • Médiane          : {serie_clean.median():.4f}")
    print(f"   • Écart-type       : {serie_clean.std():.4f}")
    print(f"   • Minimum          : {serie_clean.min():.4f}")
    print(f"   • Maximum          : {serie_clean.max():.4f}")
    print(f"   • Plage            : {(serie_clean.max() - serie_clean.min()):.4f}")
    
    # Quartiles
    q1 = serie_clean.quantile(0.25)
    q3 = serie_clean.quantile(0.75)
    print(f"   • 1er Quartile (Q1): {q1:.4f}")
    print(f"   • 3ème Quartile (Q3): {q3:.4f}")
    print(f"   • Écart interquartile (IQR): {(q3 - q1):.4f}")
    
    # Asymétrie et aplatissement
    skew_val = serie_clean.skew()
    kurt_val = serie_clean.kurtosis()
    print(f"   • Skewness         : {skew_val:.4f}")
    print(f"   • Kurtosis         : {kurt_val:.4f}")
    
    # Interprétation académique
    if abs(skew_val) < 0.5:
        skew_interp = "Distribution approximativement symétrique"
    elif skew_val > 0:
        skew_interp = "Asymétrie positive (queue à droite)"
    else:
        skew_interp = "Asymétrie négative (queue à gauche)"
    
    if kurt_val > 3:
        kurt_interp = "Distribution leptokurtique (queues épaisses)"
    elif kurt_val < 3:
        kurt_interp = "Distribution platykurtique"
    else:
        kurt_interp = "Distribution mésokurtique (proche de la normale)"
        
    print(f"     → {skew_interp}")
    print(f"     → {kurt_interp}")

# --------------------------------------------------
# 3) Analyse des variables principales
# --------------------------------------------------
print("ANALYSE DES VARIABLES PRINCIPALES :")
print("-" * 40)

variables_prioritaires = [
    'Close', 'Rendement_%', 'Volume_Log', 'Volume', 'Rendement_Log'
]

for var in variables_prioritaires:
    if var in df_clean.columns:
        libelle = {
            'Close': 'PRIX DE CLÔTURE',
            'Rendement_%': 'RENDEMENTS JOURNALIERS',
            'Volume_Log': 'VOLUME LOGARITHMIQUE',
            'Volume': 'VOLUME',
            'Rendement_Log': 'RENDEMENTS LOG'
        }
        analyse_variable_corrigee(
            f"{libelle[var]} ({var})",
            df_clean[var]
        )

# --------------------------------------------------
# 4) Résumé statistique global
# --------------------------------------------------
print("\n" + "=" * 60)
print("📋 RÉSUMÉ STATISTIQUE COMPLET (variables numériques) :")

colonnes_numeriques = df_clean.select_dtypes(include=[np.number])

if not colonnes_numeriques.empty:
    print(colonnes_numeriques.describe().round(4))
else:
    print("Aucune colonne numérique trouvée")

# --------------------------------------------------
# 5) Données manquantes
# --------------------------------------------------
print("\n" + "=" * 60)
print("🔍 INFORMATIONS SUR LES DONNÉES MANQUANTES :")
print(df_clean.isnull().sum())


In [ ]:
# ========================================
# REDÉMARRAGE COMPLET
# ========================================

print(" REDÉMARRAGE COMPLET DE L'ANALYSE")
print("=" * 50)

# 1. Télécharger à nouveau
print("\n Retéléchargement des données...")
data_new = yf.download("AAPL", start="2020-01-01", end="2024-12-01", progress=False)

# 2. Simplifier immédiatement
if isinstance(data_new.columns, pd.MultiIndex):
    data_new.columns = [col[0] for col in data_new.columns]

print(f" Données téléchargées : {len(data_new)} lignes")
print(f"Colonnes : {data_new.columns.tolist()}")

# 3. Préparation
df_clean_new = data_new.copy()
df_clean_new['Rendement_%'] = df_clean_new['Close'].pct_change() * 100
df_clean_new['Volume_Log'] = np.log(df_clean_new['Volume'] + 1)
df_clean_new = df_clean_new.dropna()

print(f"\n🔧 Préparation terminée")
print(f"   Lignes : {len(df_clean_new)}")
print(f"   Colonnes : {df_clean_new.columns.tolist()}")

# 4. Statistiques simples
print("\n" + "="*50)
print(" STATISTIQUES SIMPLES :")

print(f"\n• Prix de clôture :")
print(f"  Moyenne : ${df_clean_new['Close'].mean():.2f}")
print(f"  Médiane : ${df_clean_new['Close'].median():.2f}")

print(f"\n• Rendements :")
print(f"  Moyenne : {df_clean_new['Rendement_%'].mean():.4f}%")
print(f"  Volatilité : {df_clean_new['Rendement_%'].std():.4f}%")

# Remplacer l'ancien df_clean
df_clean = df_clean_new.copy()
print("\n Nouveau df_clean prêt pour les analyses suivantes")

In [ ]:
# ========================================
# GESTION VALEURS ABERRANTES
# ========================================

print(" GESTION DES VALEURS ABERRANTES")
print("=" * 50)

# Vérifier que df_clean existe bien
print(f" Données disponibles : {len(df_clean)} lignes")
print(f"Colonnes : {df_clean.columns.tolist()}")

# 1. Détection des outliers pour les rendements
print("\n1.  OUTLIERS DANS LES RENDEMENTS :")

returns = df_clean['Rendement_%']
    
# Méthode IQR
Q1 = returns.quantile(0.25)
Q3 = returns.quantile(0.75)
IQR = Q3 - Q1
limite_basse = Q1 - 1.5 * IQR
limite_haute = Q3 + 1.5 * IQR

outliers = returns[(returns < limite_basse) | (returns > limite_haute)]

print(f"   • Nombre d'outliers : {len(outliers)}")
print(f"   • Pourcentage : {len(outliers)/len(returns)*100:.1f}%")
print(f"   • Plage normale : [{limite_basse:.2f}%, {limite_haute:.2f}%]")

# Afficher les plus extrêmes
if len(outliers) > 0:
    print(f"   • Top 5 outliers :")
    print(outliers.sort_values(ascending=False).head().round(2))

# 2. Winsorization des rendements
print("\n2.  WINSORIZATION DES RENDEMENTS :")

from scipy.stats.mstats import winsorize

# Winsorization à 1%
df_clean['Rendement_Winsorized'] = winsorize(df_clean['Rendement_%'], limits=[0.01, 0.01])

# Comparaison
print(f"   • Avant winsorization :")
print(f"     Skewness : {df_clean['Rendement_%'].skew():.3f}")
print(f"     Kurtosis : {df_clean['Rendement_%'].kurtosis():.3f}")

print(f"   • Après winsorization :")
print(f"     Skewness : {df_clean['Rendement_Winsorized'].skew():.3f}")
print(f"     Kurtosis : {df_clean['Rendement_Winsorized'].kurtosis():.3f}")

print(f"    Colonne 'Rendement_Winsorized' ajoutée")

# 3. Vérification des données
print("\n3.  VÉRIFICATION FINALE :")
print(f"   • Lignes totales : {len(df_clean)}")
print(f"   • Valeurs manquantes : {df_clean.isnull().sum().sum()}")
print(f"   • Dernières colonnes : {list(df_clean.columns[-3:])}")

In [ ]:
# ========================================
# VISUALISATIONS UNIVARIÉES
# ========================================

print(" CRÉATION DES VISUALISATIONS")
print("=" * 50)

# Configuration des graphiques
plt.figure(figsize=(15, 10))

# 1. Série temporelle du prix
plt.subplot(2, 3, 1)
plt.plot(df_clean.index, df_clean['Close'], color='blue', linewidth=1)
plt.title('Apple - Évolution du Prix (2020-2024)', fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Prix ($)')
plt.grid(True, alpha=0.3)
plt.tick_params(axis='x', rotation=45)

# 2. Histogramme du prix
plt.subplot(2, 3, 2)
plt.hist(df_clean['Close'], bins=40, edgecolor='black', alpha=0.7, color='skyblue')
plt.title('Distribution des Prix', fontweight='bold')
plt.xlabel('Prix ($)')
plt.ylabel('Fréquence')
plt.axvline(df_clean['Close'].mean(), color='red', linestyle='--', label='Moyenne')
plt.legend()

# 3. QQ-Plot du prix
plt.subplot(2, 3, 3)
stats.probplot(df_clean['Close'], dist="norm", plot=plt)
plt.title('QQ-Plot: Prix vs Loi Normale', fontweight='bold')

# 4. Série des rendements
plt.subplot(2, 3, 4)
plt.plot(df_clean.index, df_clean['Rendement_%'], color='green', linewidth=0.5, alpha=0.7)
plt.axhline(y=0, color='red', linestyle='-', alpha=0.5)
plt.title('Rendements Journaliers', fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Rendement (%)')
plt.tick_params(axis='x', rotation=45)

# 5. Histogramme des rendements
plt.subplot(2, 3, 5)
plt.hist(df_clean['Rendement_%'], bins=50, edgecolor='black', alpha=0.7, color='lightgreen')
plt.title('Distribution des Rendements', fontweight='bold')
plt.xlabel('Rendement (%)')
plt.ylabel('Fréquence')

# 6. Boxplot des rendements
plt.subplot(2, 3, 6)
plt.boxplot(df_clean['Rendement_%'], vert=True)
plt.title('Boxplot des Rendements', fontweight='bold')
plt.ylabel('Rendement (%)')

plt.suptitle('ANALYSE UNIVARIÉE - APPLE (AAPL)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('analyse_univariee_finale.png', dpi=300, bbox_inches='tight')
plt.show()

print(" Graphiques sauvegardés dans 'analyse_univariee_finale.png'")

In [ ]:
# ========================================
# ÉTAPE 3.1 : CORRÉLATIONS ENTRE VARIABLES EXPLICATIVES
# ========================================

print(" ÉTAPE 3.1 : CORRÉLATIONS ENTRE VARIABLES EXPLICATIVES")
print("=" * 60)

# Création de variables supplémentaires pour l'analyse
print("\n Création de variables techniques...")

# Moyenne mobile 20 jours
df_clean['MA_20'] = df_clean['Close'].rolling(window=20).mean()

# Volatilité historique (20 jours)
df_clean['Volatilite_20'] = df_clean['Rendement_%'].rolling(window=20).std()

# Supprimer les NaN dus aux rolling windows
df_analysis = df_clean.dropna().copy()

print(f" Variables créées")
print(f"   • Lignes pour analyse : {len(df_analysis)}")
print(f"   • Variables disponibles : {df_analysis.columns.tolist()}")

# 1. Matrice de corrélation de Pearson
print("\n1.  MATRICE DE CORRÉLATION DE PEARSON :")

# Sélection des variables pour l'analyse
variables_correlation = ['Close', 'Volume', 'Rendement_%', 'MA_20', 'Volatilite_20']

# Calcul de la matrice
corr_matrix = df_analysis[variables_correlation].corr()

print("Matrice de corrélation :")
print(corr_matrix.round(3))

# 2. Heatmap de visualisation
print("\n2.  VISUALISATION DE LA MATRICE DE CORRÉLATION :")

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, 
            annot=True, 
            cmap='coolwarm', 
            center=0,
            fmt='.2f',
            square=True,
            linewidths=1,
            cbar_kws={"shrink": 0.8})
plt.title('Matrice de Corrélation entre Variables Explicatives', fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('matrice_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. Analyse des corrélations fortes
print("\n3.  ANALYSE DES CORRÉLATIONS SIGNIFICATIVES :")

seuil_fort = 0.7  # Corrélation forte
seuil_modere = 0.5  # Corrélation modérée

print("Corrélations fortes (|ρ| > 0.7) :")
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr = corr_matrix.iloc[i, j]
        if abs(corr) > seuil_fort:
            print(f"   • {corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]} : {corr:.3f}")
            if corr > 0:
                print(f"     → Forte corrélation POSITIVE")
            else:
                print(f"     → Forte corrélation NÉGATIVE")

print("\nCorrélations modérées (0.5 < |ρ| < 0.7) :")
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr = corr_matrix.iloc[i, j]
        if seuil_modere < abs(corr) <= seuil_fort:
            print(f"   • {corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]} : {corr:.3f}")

# ========================================
#   ÉTAPE 3 : ANALYSE BIVARIÉE
# ========================================

### II-3-1 Entre les variables explicatives :
1. Calcul du coefficient de corrélation de Pearson
2. Visualiser la relation entre variables  
3. Faire les transformations nécessaires
4. Analyser les sous-groupes de variables corrélées

### II-3-2 Entre variable à expliquer Y et variables explicatives :
1. Calcul du coefficient de corrélation de Pearson
2. Faire le Test ANOVA
3. Faire le test de Student
4. Distinguer variables significatives et non significatives

In [ ]:
# ========================================
# PRÉPARATION POUR L'ANALYSE BIVARIÉE
# ========================================

print(" PRÉPARATION POUR L'ANALYSE BIVARIÉE")
print("=" * 60)

# Vérifier que df_clean existe
print("Vérification des données disponibles...")
print(f"df_clean existe ? {'df_clean' in dir()}")
if 'df_clean' in dir():
    print(f"Shape de df_clean : {df_clean.shape}")
    print(f"Colonnes : {df_clean.columns.tolist()}")

# Création de variables supplémentaires
print("\n Création de variables supplémentaires :")

# Moyenne mobile 20 jours
df_clean['MA_20'] = df_clean['Close'].rolling(window=20).mean()

# Volatilité historique (20 jours)
df_clean['Volatilite_20'] = df_clean['Rendement_%'].rolling(window=20).std()

# Momentum (rendement sur 5 jours)
df_clean['Rendement_5j'] = df_clean['Close'].pct_change(periods=5) * 100

# Ratio volume / moyenne volume
df_clean['Volume_Ratio'] = df_clean['Volume'] / df_clean['Volume'].rolling(window=20).mean()

# Supprimer les NaN dus aux calculs
df_bivarie = df_clean.dropna().copy()

print(f" Préparation terminée")
print(f"   • Observations : {len(df_bivarie)}")
print(f"   • Nouvelles variables : MA_20, Volatilite_20, Rendement_5j, Volume_Ratio")
print(f"   • Données prêtes pour l'analyse bivariée")

In [ ]:
# ========================================
# II-3-1 : CORRELATIONS ENTRE VARIABLES EXPLICATIVES
# ========================================

print("II-3-1 : CORRELATIONS ENTRE VARIABLES EXPLICATIVES")
print("=" * 60)

# Selection des variables pour l'analyse de correlation
variables_correlation = ['Close', 'Volume_Log', 'Rendement_%', 'MA_20', 
                        'Volatilite_20', 'Rendement_5j', 'Volume_Ratio']

print(f"Variables analysees : {variables_correlation}")

# 1. Calcul de la matrice de correlation de Pearson
print("\n1. MATRICE DE CORRELATION DE PEARSON :")
corr_matrix = df_bivarie[variables_correlation].corr()

# Affichage de la matrice
print(corr_matrix.round(3))

# 2. Visualisation par heatmap
print("\n2. VISUALISATION DE LA MATRICE DE CORRELATION :")

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, 
            annot=True, 
            cmap='coolwarm', 
            center=0,
            fmt='.2f',
            square=True,
            linewidths=1,
            cbar_kws={"shrink": 0.8},
            annot_kws={"size": 10})

plt.title('Matrice de Correlation de Pearson entre Variables Explicatives', 
          fontsize=14, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('correlation_pearson.png', dpi=300, bbox_inches='tight')
plt.show()

print("Graphique sauvegarde : 'correlation_pearson.png'")

# 3. Analyse des correlations fortes
print("\n3. ANALYSE DES CORRELATIONS SIGNIFICATIVES :")

seuil_fort = 0.7
seuil_modere = 0.5

print("a) Correlations FORTES (|ρ| > 0.7) :")
fortes_correlations = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr = corr_matrix.iloc[i, j]
        if abs(corr) > seuil_fort:
            print(f"   {corr_matrix.columns[i]} <-> {corr_matrix.columns[j]} : {corr:.3f}")
            if corr > 0:
                interpretation = "Forte correlation POSITIVE"
            else:
                interpretation = "Forte correlation NEGATIVE"
            print(f"     {interpretation}")
            fortes_correlations.append((corr_matrix.columns[i], corr_matrix.columns[j], corr))

print("\nb) Correlations MODEREES (0.5 < |ρ| ≤ 0.7) :")
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr = corr_matrix.iloc[i, j]
        if seuil_modere < abs(corr) <= seuil_fort:
            print(f"   {corr_matrix.columns[i]} <-> {corr_matrix.columns[j]} : {corr:.3f}")

# 4. Detection de multicolinearite
print("\n4. DETECTION DE MULTICOLINEARITE :")

# Calcul du VIF (Variance Inflation Factor)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# Preparation pour VIF
X = add_constant(df_bivarie[variables_correlation])
vif_data = pd.DataFrame()
vif_data["Variable"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print("Facteur d'Inflation de la Variance (VIF) :")
print(vif_data.round(2))

print("\nRegle d'interpretation :")
print("VIF < 5 : Pas de multicolinearite problematique")
print("5 ≤ VIF < 10 : Multicolinearite moderee")
print("VIF ≥ 10 : Multicolinearite severe")

In [ ]:
# ========================================
# VISUALISATION DES RELATIONS BIVARIEES
# ========================================

print("VISUALISATION DES RELATIONS ENTRE VARIABLES")
print("=" * 60)

# Selection des paires les plus interessantes
paires_importantes = [
    ('Close', 'Volume_Log'),
    ('Rendement_%', 'Volatilite_20'),
    ('Close', 'MA_20'),
    ('Volume_Log', 'Volume_Ratio'),
    ('Rendement_%', 'Rendement_5j'),
    ('Close', 'Volatilite_20')
]

# Creation des scatter plots avec regressions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (var_x, var_y) in enumerate(paires_importantes):
    ax = axes[idx]
    
    # Nuage de points
    scatter = ax.scatter(df_bivarie[var_x], df_bivarie[var_y], 
                        alpha=0.6, s=30, 
                        c='steelblue', 
                        edgecolors='white', linewidth=0.5)
    
    # Calcul de la correlation
    corr_value = df_bivarie[var_x].corr(df_bivarie[var_y])
    
    # Regression lineaire (si relation lineaire)
    if abs(corr_value) > 0.2:  # Seuil minimal pour tenter une regression
        try:
            # Regression polynomiale de degre 1 (lineaire)
            z = np.polyfit(df_bivarie[var_x], df_bivarie[var_y], 1)
            p = np.poly1d(z)
            
            # Tracer la droite de regression
            x_range = np.linspace(df_bivarie[var_x].min(), df_bivarie[var_x].max(), 100)
            ax.plot(x_range, p(x_range), "r-", linewidth=2, 
                    label=f'y = {z[0]:.3f}x + {z[1]:.2f}')
            
            # Coefficient de determination R²
            y_pred = p(df_bivarie[var_x])
            ss_res = np.sum((df_bivarie[var_y] - y_pred) ** 2)
            ss_tot = np.sum((df_bivarie[var_y] - df_bivarie[var_y].mean()) ** 2)
            r_squared = 1 - (ss_res / ss_tot)
            
            regression_info = f'R² = {r_squared:.3f}'
        except:
            regression_info = "Regression non calculable"
    else:
        regression_info = "Correlation trop faible"
    
    # Configuration du graphique
    ax.set_xlabel(var_x, fontsize=11)
    ax.set_ylabel(var_y, fontsize=11)
    ax.set_title(f'{var_y} vs {var_x}', fontweight='bold', fontsize=12)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Legende avec informations
    info_text = f'ρ = {corr_value:.3f}\n{regression_info}'
    ax.text(0.05, 0.95, info_text, 
            transform=ax.transAxes, fontsize=10,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    if abs(corr_value) > 0.2:
        ax.legend(loc='lower right', fontsize=9)

plt.suptitle('VISUALISATION DES RELATIONS BIVARIEES ENTRE VARIABLES EXPLICATIVES', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('relations_bivariees_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

print("Graphiques des relations bivariees sauvegardes")

In [ ]:
# ========================================
# TRANSFORMATIONS DES RELATIONS NON-LINEAIRES
# ========================================

print("TRANSFORMATIONS DES RELATIONS NON-LINEAIRES")
print("=" * 60)

# 1. Identification des relations non-lineaires
print("1. IDENTIFICATION DES RELATIONS NON-LINEAIRES :")

# Test de non-linearite par regression polynomiale
import statsmodels.api as sm
variables_test = ['Close', 'Volume_Log', 'Rendement_%', 'Volatilite_20']

print("Test de linearite (comparaison modele lineaire vs quadratique) :")
print("-" * 70)

for var_x in variables_test:
    for var_y in variables_test:
        if var_x != var_y:
            # Eviter les doublons
            if variables_test.index(var_x) < variables_test.index(var_y):
                # Modele lineaire
                X_lin = df_bivarie[[var_x]].assign(const=1)
                model_lin = sm.OLS(df_bivarie[var_y], X_lin).fit()
                r2_lin = model_lin.rsquared
                
                # Modele quadratique
                df_bivarie[f'{var_x}_sq'] = df_bivarie[var_x] ** 2
                X_quad = df_bivarie[[var_x, f'{var_x}_sq']].assign(const=1)
                model_quad = sm.OLS(df_bivarie[var_y], X_quad).fit()
                r2_quad = model_quad.rsquared
                
                # Test si le modele quadratique est significativement meilleur
                improvement = r2_quad - r2_lin
                
                if improvement > 0.05:  # Amelioration de plus de 5% en R²
                    print(f"{var_y} vs {var_x} :")
                    print(f"  R² lineaire = {r2_lin:.3f}, R² quadratique = {r2_quad:.3f}")
                    print(f"  Amelioration = {improvement:.3f} -> RELATION NON-LINEAIRE DETECTEE")
                    print(f"  Transformation suggeree : ajouter {var_x}² au modele")
                    print()

# 2. Transformation logarithmique pour certaines variables
print("\n2. TRANSFORMATIONS APPLICABLES :")

# Verifier la distribution des variables
print("Analyse de la distribution avant transformation :")
for var in ['Volume', 'Volume_Ratio', 'Volatilite_20']:
    if var in df_bivarie.columns:
        skewness = df_bivarie[var].skew()
        print(f"{var:15} : Skewness = {skewness:.3f}", end=" ")
        if abs(skewness) > 1:
            print("-> Forte asymetrie, transformation log recommandee")
        elif abs(skewness) > 0.5:
            print("-> Asymetrie moderee, transformation possible")
        else:
            print("-> Distribution relativement symetrique")

# 3. Application des transformations
print("\n3. APPLICATION DES TRANSFORMATIONS :")

# Transformation Box-Cox pour normaliser
from scipy import stats

variables_a_transformer = ['Volume_Ratio', 'Volatilite_20']

for var in variables_a_transformer:
    if var in df_bivarie.columns:
        # Verifier que toutes les valeurs sont positives (requis pour Box-Cox)
        if (df_bivarie[var] > 0).all():
            try:
                # Transformation Box-Cox
                transformed_data, lambda_value = stats.boxcox(df_bivarie[var])
                df_bivarie[f'{var}_BoxCox'] = transformed_data
                
                print(f"{var:15} : Transformation Box-Cox appliquee (λ = {lambda_value:.3f})")
                print(f"  Skewness avant : {df_bivarie[var].skew():.3f}")
                print(f"  Skewness apres : {df_bivarie[f'{var}_BoxCox'].skew():.3f}")
            except:
                # Fallback sur transformation log
                df_bivarie[f'{var}_Log'] = np.log1p(df_bivarie[var])
                print(f"{var:15} : Transformation logarithmique appliquee")
        else:
            # Transformation log1p (log(1+x)) pour gerer les zeros
            df_bivarie[f'{var}_Log'] = np.log1p(df_bivarie[var] - df_bivarie[var].min() + 1)
            print(f"{var:15} : Transformation log1p appliquee")

print("\nNouvelles variables transformees creees")

In [ ]:
# ========================================
# II-3-2 : RELATIONS AVEC LA VARIABLE CIBLE Y
# ========================================

print("II-3-2 : ANALYSE AVEC LA VARIABLE CIBLE Y")
print("=" * 60)

# 1. Definition de la variable cible Y
print("\n1. DEFINITION DE LA VARIABLE CIBLE :")

# Option 1 : Classification binaire (direction du prix)
df_bivarie['Y_Classification'] = np.where(df_bivarie['Rendement_%'].shift(-1) > 0, 1, 0)

# Option 2 : Regression (magnitude du rendement futur)
df_bivarie['Y_Regression'] = df_bivarie['Rendement_%'].shift(-1)

# Option 3 : Categorielle multi-classes
df_bivarie['Y_Categories'] = pd.cut(df_bivarie['Rendement_%'].shift(-1), 
                                    bins=[-np.inf, -1, 1, np.inf], 
                                    labels=['Baisse', 'Neutre', 'Hausse'])

# Supprimer la derniere ligne (NaN pour les targets)
df_target = df_bivarie.dropna().copy()

print(f"Variables cibles creees :")
print(f"   Y_Classification : 1 = hausse demain, 0 = baisse/stagnation")
print(f"   Y_Regression : Rendement futur en % (probleme de regression)")
print(f"   Y_Categories : 3 classes (Baisse, Neutre, Hausse)")
print(f"   Observations disponibles : {len(df_target)}")

# 2. Correlations avec la variable cible
print("\n2. CORRELATIONS DE PEARSON AVEC LA CIBLE :")

variables_explicatives = ['Close', 'Volume_Log', 'Rendement_%', 'MA_20', 
                         'Volatilite_20', 'Rendement_5j', 'Volume_Ratio']

print("a) Avec Y_Regression (variable continue) :")
print("-" * 50)
correlations_regression = []
for var in variables_explicatives:
    if var in df_target.columns:
        corr = df_target[var].corr(df_target['Y_Regression'])
        correlations_regression.append((var, corr))
        print(f"{var:15} : ρ = {corr:.4f}")

# Trier par correlation absolue
correlations_regression.sort(key=lambda x: abs(x[1]), reverse=True)
print(f"\nVariables les plus correlees (en valeur absolue) :")
for var, corr in correlations_regression[:3]:
    print(f"  {var:15} : |ρ| = {abs(corr):.4f}")

print("\nb) Avec Y_Classification (variable binaire - correlation point-biseriale) :")
print("-" * 50)
correlations_classification = []
for var in variables_explicatives:
    if var in df_target.columns:
        corr = df_target[var].corr(df_target['Y_Classification'])
        correlations_classification.append((var, corr))
        print(f"{var:15} : r_pb = {corr:.4f}")

# 3. Test ANOVA (pour variables categorielles)
print("\n3. TEST ANOVA :")
print("Analyse de la variance des rendements selon la categorie future :")

# Preparation des groupes pour ANOVA
groupes_anova = []
for categorie in df_target['Y_Categories'].cat.categories:
    groupe = df_target[df_target['Y_Categories'] == categorie]['Rendement_%']
    groupes_anova.append(groupe)
    print(f"{categorie:10} : n = {len(groupe)}, moyenne = {groupe.mean():.4f}%")

# Test ANOVA
f_stat, p_value = stats.f_oneway(*groupes_anova)

print(f"\nResultats ANOVA :")
print(f"F-statistic = {f_stat:.4f}")
print(f"p-value = {p_value:.6f}")

if p_value < 0.05:
    print("DIFFERENCE SIGNIFICATIVE entre les groupes (p < 0.05)")
    print("La categorie de rendement futur influence le rendement actuel")
else:
    print("Pas de difference significative entre les groupes")
    print("La categorie de rendement futur n'a pas d'impact detectable")

# 4. Test de Student (t-test)
print("\n4. TEST DE STUDENT :")
print("Comparaison des moyennes entre jours precedant une hausse vs une baisse :")

# Separation en deux groupes
groupe_hausse = df_target[df_target['Y_Classification'] == 1]
groupe_baisse = df_target[df_target['Y_Classification'] == 0]

print(f"Groupe 'Hausse demain' : {len(groupe_hausse)} observations")
print(f"Groupe 'Baisse/Stagnation demain' : {len(groupe_baisse)} observations")

# Tests t pour chaque variable explicative
print("\nResultats des tests t (comparaison des moyennes) :")
print("-" * 70)
print(f"{'Variable':<15} {'Moyenne Hausse':<15} {'Moyenne Baisse':<15} {'t-stat':<10} {'p-value':<10} {'Significatif':<12}")
print("-" * 70)

variables_t_test = ['Rendement_%', 'Volatilite_20', 'Volume_Log', 'Rendement_5j']

for var in variables_t_test:
    if var in df_target.columns:
        # Donnees des deux groupes
        data_hausse = groupe_hausse[var].dropna()
        data_baisse = groupe_baisse[var].dropna()
        
        # Test t independant (pas d'hypothese d'egalite des variances)
        t_stat, p_val = stats.ttest_ind(data_hausse, data_baisse, equal_var=False)
        
        # Moyennes
        mean_hausse = data_hausse.mean()
        mean_baisse = data_baisse.mean()
        
        # Interpretation
        significatif = "OUI" if p_val < 0.05 else "NON"
        
        print(f"{var:<15} {mean_hausse:>12.4f}   {mean_baisse:>12.4f}   {t_stat:>8.3f}   {p_val:>8.4f}   {significatif:^12}")

print("-" * 70)

# 5. Analyse des variables significatives
print("\n5. SYNTHESE DES VARIABLES SIGNIFICATIVES :")

variables_significatives = []
variables_non_significatives = []

for var in variables_t_test:
    if var in df_target.columns:
        data_hausse = groupe_hausse[var].dropna()
        data_baisse = groupe_baisse[var].dropna()
        _, p_val = stats.ttest_ind(data_hausse, data_baisse, equal_var=False)
        
        if p_val < 0.05:
            variables_significatives.append((var, p_val))
        else:
            variables_non_significatives.append((var, p_val))

print("Variables SIGNIFICATIVES (p < 0.05) :")
if variables_significatives:
    for var, p_val in variables_significatives:
        print(f"  {var:15} : p-value = {p_val:.4f}")
else:
    print("  Aucune variable significative au seuil de 5%")

print("\nVariables NON SIGNIFICATIVES :")
for var, p_val in variables_non_significatives:
    print(f"  {var:15} : p-value = {p_val:.4f}")

print("\nAnalyse bivariee avec la variable cible terminee")

# ========================================
#   ÉTAPE 4 : ANALYSE MULTIVARIÉE
# ========================================

### II-4-1 Cas de variables numériques :
1. Vérifier les conditions d'application de l'ACP
2. Vérifier s'il y a des variables ou individus supplémentaires
3. Appliquer l'ACP pour atteindre 80% de variance sur les 2 premières composantes
4. Interpréter les résultats de projection

### Objectifs pour notre projet finance :
- Réduire la dimensionnalité des données
- Identifier les principales composantes explicatives
- Préparer les données pour la modélisation ML

In [ ]:
# ========================================
# APPLICATION DE L'ACP 
# ========================================

print("APPLICATION DE L'ACP - CORRECTION DE LA MULTICOLINEARITE")
print("=" * 80)

# 1. Suppression des variables problématiques
print("1. CORRECTION DE LA MULTICOLINEARITE :")

# Supprimer MA_20 (trop corrélé avec Close)
variables_corrigees = ['Close', 'Volume_Log', 'Rendement_%', 
                      'Volatilite_20', 'Rendement_5j', 'Volume_Ratio']

print(f"Variables après correction : {variables_corrigees}")
print(f"Variable supprimée : MA_20 (VIF = 89.24, corrélation avec Close = 0.99)")

X_acp_corr = df_bivarie[variables_corrigees].copy()

# 2. Vérification rapide après correction
print("\n2. VERIFICATION RAPIDE APRES CORRECTION :")

# Recalcul du VIF
X_for_vif_corr = add_constant(X_acp_corr)
vif_data_corr = pd.DataFrame()
vif_data_corr["Variable"] = X_for_vif_corr.columns
vif_data_corr["VIF"] = [variance_inflation_factor(X_for_vif_corr.values, i) 
                        for i in range(X_for_vif_corr.shape[1])]
vif_data_corr = vif_data_corr[vif_data_corr["Variable"] != "const"]

print("Nouveaux VIF après suppression de MA_20 :")
print(vif_data_corr.round(2))

# Vérification des VIF problématiques
vif_problematique_corr = vif_data_corr[vif_data_corr["VIF"] >= 10]
print(f"\nVariables avec VIF >= 10 : {len(vif_problematique_corr)}")

if len(vif_problematique_corr) == 0:
    print("✅ Multicolinéarité corrigée")
else:
    print(f"⚠️  Variables encore problématiques : {list(vif_problematique_corr['Variable'])}")

# 3. Standardisation des données
print("\n3. STANDARDISATION DES DONNEES :")

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_acp_corr)

print(f"Données standardisées : {X_scaled.shape}")
print(f"Moyenne après standardisation : {np.mean(X_scaled):.6f}")
print(f"Écart-type après standardisation : {np.std(X_scaled):.6f}")

# 4. Application de l'ACP
print("\n4. APPLICATION DE L'ACP :")

from sklearn.decomposition import PCA

# Appliquer l'ACP
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

print(f"Nombre de composantes : {pca.n_components_}")
print(f"Variance totale expliquée : {np.sum(pca.explained_variance_ratio_):.4f}")

# 5. Analyse de la variance expliquée
print("\n5. ANALYSE DE LA VARIANCE EXPLIQUEE :")

variance_expliquee = pca.explained_variance_ratio_
variance_cumulee = np.cumsum(variance_expliquee)

print("Variance expliquée par composante :")
for i, (var, cum_var) in enumerate(zip(variance_expliquee, variance_cumulee)):
    print(f"   Composante {i+1:2d} : {var:.4f} ({var*100:6.2f}%) | Cumulé : {cum_var:.4f}")

# Trouver le nombre de composantes pour 80% de variance
n_composantes_80 = np.argmax(variance_cumulee >= 0.80) + 1
print(f"\nComposantes nécessaires pour 80% de variance : {n_composantes_80}")
print(f"Variance cumulée à {n_composantes_80} composantes : {variance_cumulee[n_composantes_80-1]:.4f}")

# 6. Graphique de la variance expliquée
print("\n6. VISUALISATION DE LA VARIANCE EXPLIQUEE :")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique en barre (éboulis)
ax1 = axes[0]
ax1.bar(range(1, len(variance_expliquee)+1), variance_expliquee * 100, 
        alpha=0.7, color='steelblue', label='Variance individuelle')
ax1.plot(range(1, len(variance_cumulee)+1), variance_cumulee * 100, 
         'ro-', linewidth=2, markersize=8, label='Variance cumulée')
ax1.axhline(y=80, color='green', linestyle='--', alpha=0.7, label='Seuil 80%')
ax1.axvline(x=n_composantes_80, color='red', linestyle='--', alpha=0.7)
ax1.set_xlabel('Composantes principales')
ax1.set_ylabel('Variance expliquée (%)')
ax1.set_title('Graphique d\'éboulis des valeurs propres', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Seuil de Kaiser (valeur propre > 1)
ax2 = axes[1]
valeurs_propres = pca.explained_variance_
ax2.plot(range(1, len(valeurs_propres)+1), valeurs_propres, 'bo-', linewidth=2, markersize=8)
ax2.axhline(y=1, color='red', linestyle='--', alpha=0.7, label='Seuil de Kaiser (λ=1)')
ax2.set_xlabel('Composantes principales')
ax2.set_ylabel('Valeurs propres (λ)')
ax2.set_title('Valeurs propres (critère de Kaiser)', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('ANALYSE DE LA VARIANCE EXPLIQUEE PAR L\'ACP', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('acp_variance_expliquee_corrigee.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Graphique sauvegardé : 'acp_variance_expliquee.png'")

In [ ]:
# ========================================
# INTERPRETATION DES COMPOSANTES PRINCIPALES
# ========================================

print("INTERPRETATION DES COMPOSANTES PRINCIPALES")
print("=" * 80)

# 1. Matrice des composantes (loadings)
print("1. MATRICE DES COMPOSANTES (LOADINGS) :")

# Création d'un DataFrame pour les loadings
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)],
    index=X_acp_corr.columns
)

print("Loadings des 3 premières composantes (80% de variance) :")
print(loadings[['PC1', 'PC2', 'PC3']].round(3))

# 2. Contribution des variables aux composantes
print("\n2. CONTRIBUTION DES VARIABLES AUX COMPOSANTES :")

print("Variables importantes pour chaque composante (|loading| > 0.5) :")

# PC1
print(f"\n• PC1 (38.8% de variance) :")
pc1_important = loadings['PC1'][abs(loadings['PC1']) > 0.5]
if len(pc1_important) > 0:
    for var, loading in pc1_important.sort_values(key=abs, ascending=False).items():
        sign = "POSITIF" if loading > 0 else "NEGATIF"
        print(f"  {var:15} : {loading:.3f} ({sign})")
else:
    print("  Aucune variable très fortement corrélée")

# PC2
print(f"\n• PC2 (23.8% de variance) :")
pc2_important = loadings['PC2'][abs(loadings['PC2']) > 0.5]
if len(pc2_important) > 0:
    for var, loading in pc2_important.sort_values(key=abs, ascending=False).items():
        sign = "POSITIF" if loading > 0 else "NEGATIF"
        print(f"  {var:15} : {loading:.3f} ({sign})")
else:
    print("  Aucune variable très fortement corrélée")

# PC3
print(f"\n• PC3 (19.2% de variance) :")
pc3_important = loadings['PC3'][abs(loadings['PC3']) > 0.5]
if len(pc3_important) > 0:
    for var, loading in pc3_important.sort_values(key=abs, ascending=False).items():
        sign = "POSITIF" if loading > 0 else "NEGATIF"
        print(f"  {var:15} : {loading:.3f} ({sign})")
else:
    print("  Aucune variable très fortement corrélée")

# 3. Interprétation qualitative
print("\n3. INTERPRETATION QUALITATIVE DES COMPOSANTES :")

print("Analyse détaillée des loadings :")

# PC1 - Analyse
print(f"\n PC1 (38.8% de variance) :")
pc1_analysis = loadings['PC1'].sort_values(ascending=False)
print("Variables les plus influentes :")
for var, loading in pc1_analysis.head(3).items():
    if abs(loading) > 0.3:
        interpretation = ""
        if var == 'Close':
            interpretation = "→ Représente le NIVEAU DE PRIX"
        elif var == 'Volume_Log':
            interpretation = "→ Représente le VOLUME d'échanges"
        elif var == 'Volatilite_20':
            interpretation = "→ Représente la VOLATILITÉ"
        print(f"  {var:15} : {loading:.3f} {interpretation}")

# Tendances dans PC1
pos_vars_pc1 = [var for var, val in pc1_analysis.items() if val > 0.3]
neg_vars_pc1 = [var for var, val in pc1_analysis.items() if val < -0.3]

if pos_vars_pc1:
    print(f"  Variables POSITIVES : {pos_vars_pc1}")
if neg_vars_pc1:
    print(f"  Variables NEGATIVES : {neg_vars_pc1}")

print(f"  INTERPRETATION : PC1 semble représenter une dimension de 'NIVEAU DE MARCHE'")

# PC2 - Analyse
print(f"\n PC2 (23.8% de variance) :")
pc2_analysis = loadings['PC2'].sort_values(ascending=False)
print("Variables les plus influentes :")
for var, loading in pc2_analysis.head(3).items():
    if abs(loading) > 0.3:
        interpretation = ""
        if var == 'Rendement_%':
            interpretation = "→ Représente le RENDEMENT COURANT"
        elif var == 'Rendement_5j':
            interpretation = "→ Représente le MOMENTUM (5 jours)"
        elif var == 'Volume_Ratio':
            interpretation = "→ Représente l'ACTIVITE RELATIVE"
        print(f"  {var:15} : {loading:.3f} {interpretation}")

# Tendances dans PC2
pos_vars_pc2 = [var for var, val in pc2_analysis.items() if val > 0.3]
neg_vars_pc2 = [var for var, val in pc2_analysis.items() if val < -0.3]

if pos_vars_pc2:
    print(f"  Variables POSITIVES : {pos_vars_pc2}")
if neg_vars_pc2:
    print(f"  Variables NEGATIVES : {neg_vars_pc2}")

print(f"  INTERPRETATION : PC2 semble représenter une dimension de 'MOMENTUM & ACTIVITE'")

# PC3 - Analyse
print(f"\n PC3 (19.2% de variance) :")
pc3_analysis = loadings['PC3'].sort_values(ascending=False)
print("Variables les plus influentes :")
for var, loading in pc3_analysis.head(3).items():
    if abs(loading) > 0.3:
        interpretation = ""
        if var == 'Volume_Ratio':
            interpretation = "→ Représente l'ACTIVITE ANORMALE"
        elif var == 'Volatilite_20':
            interpretation = "→ Représente le RISQUE HISTORIQUE"
        elif var == 'Rendement_5j':
            interpretation = "→ Représente la TENDANCE"
        print(f"  {var:15} : {loading:.3f} {interpretation}")

print(f"  INTERPRETATION : PC3 semble représenter une dimension de 'RISQUE & ACTIVITE ANORMALE'")

# 4. Cercle des corrélations
print("\n4. VISUALISATION - CERCLES DES CORRELATIONS :")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Premier plan factoriel (PC1 vs PC2)
ax1 = axes[0]
for var in X_acp_corr.columns:
    ax1.arrow(0, 0, loadings.loc[var, 'PC1'], loadings.loc[var, 'PC2'], 
              head_width=0.03, head_length=0.03, fc='blue', ec='blue', alpha=0.7)
    ax1.text(loadings.loc[var, 'PC1']*1.1, loadings.loc[var, 'PC2']*1.1, 
             var, fontsize=9, ha='center', va='center')

# Cercle unité
circle = plt.Circle((0,0), 1, color='gray', fill=False, linestyle='--', alpha=0.5)
ax1.add_artist(circle)

ax1.set_xlabel(f'PC1 ({variance_expliquee[0]*100:.1f}%)', fontweight='bold')
ax1.set_ylabel(f'PC2 ({variance_expliquee[1]*100:.1f}%)', fontweight='bold')
ax1.set_title('Cercle des corrélations : PC1 vs PC2', fontweight='bold')
ax1.set_xlim(-1.2, 1.2)
ax1.set_ylim(-1.2, 1.2)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax1.axvline(x=0, color='black', linestyle='-', alpha=0.3)

# Second plan factoriel (PC1 vs PC3)
ax2 = axes[1]
for var in X_acp_corr.columns:
    ax2.arrow(0, 0, loadings.loc[var, 'PC1'], loadings.loc[var, 'PC3'], 
              head_width=0.03, head_length=0.03, fc='red', ec='red', alpha=0.7)
    ax2.text(loadings.loc[var, 'PC1']*1.1, loadings.loc[var, 'PC3']*1.1, 
             var, fontsize=9, ha='center', va='center')

circle2 = plt.Circle((0,0), 1, color='gray', fill=False, linestyle='--', alpha=0.5)
ax2.add_artist(circle2)

ax2.set_xlabel(f'PC1 ({variance_expliquee[0]*100:.1f}%)', fontweight='bold')
ax2.set_ylabel(f'PC3 ({variance_expliquee[2]*100:.1f}%)', fontweight='bold')
ax2.set_title('Cercle des corrélations : PC1 vs PC3', fontweight='bold')
ax2.set_xlim(-1.2, 1.2)
ax2.set_ylim(-1.2, 1.2)
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax2.axvline(x=0, color='black', linestyle='-', alpha=0.3)

plt.suptitle('CERCLES DES CORRELATIONS - COMPOSANTES PRINCIPALES', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('acp_cercle_correlations_final.png', dpi=300, bbox_inches='tight')
plt.show()

print(" Graphique sauvegardé : 'acp_cercle_correlations_final.png'")

In [ ]:
# ========================================
# PROJECTION DES INDIVIDUS ET SYNTHESE FINALE
# ========================================

print("PROJECTION DES INDIVIDUS ET SYNTHESE FINALE DE L'ACP")
print("=" * 80)

# 1. Création des scores des composantes
print("1. CREATION DES SCORES DES COMPOSANTES PRINCIPALES :")

# Conserver les 3 premières composantes (80% de variance)
n_composantes = 3
composantes_scores = pd.DataFrame(
    X_pca[:, :n_composantes],
    columns=[f'PC{i+1}' for i in range(n_composantes)],
    index=X_acp_corr.index
)

# Ajouter au DataFrame principal
df_bivarie['PC1'] = composantes_scores['PC1']
df_bivarie['PC2'] = composantes_scores['PC2']
df_bivarie['PC3'] = composantes_scores['PC3']

print(f"Composantes principales ajoutées au DataFrame :")
print(f"• PC1 : {composantes_scores['PC1'].mean():.3f} ± {composantes_scores['PC1'].std():.3f}")
print(f"• PC2 : {composantes_scores['PC2'].mean():.3f} ± {composantes_scores['PC2'].std():.3f}")
print(f"• PC3 : {composantes_scores['PC3'].mean():.3f} ± {composantes_scores['PC3'].std():.3f}")
print(f"• Variance totale expliquée : {variance_cumulee[2]:.1%}")

# 2. Projection des individus
print("\n2. PROJECTION DES INDIVIDUS DANS LE PREMIER PLAN FACTORIEL :")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Coloriage par année
ax1 = axes[0]
scatter1 = ax1.scatter(composantes_scores['PC1'], composantes_scores['PC2'],
                      c=composantes_scores.index.year,
                      cmap='viridis',
                      alpha=0.6,
                      s=30)

ax1.set_xlabel(f'PC1 ({variance_expliquee[0]*100:.1f}%)', fontweight='bold')
ax1.set_ylabel(f'PC2 ({variance_expliquee[1]*100:.1f}%)', fontweight='bold')
ax1.set_title('Projection des jours de trading (coloriés par année)', fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax1.axvline(x=0, color='black', linestyle='-', alpha=0.3)

# Colorbar pour les années
cbar1 = plt.colorbar(scatter1, ax=ax1)
cbar1.set_label('Année')

# Coloriage par rendement futur
ax2 = axes[1]
if 'Y_Regression' in df_bivarie.columns:
    scatter2 = ax2.scatter(composantes_scores['PC1'], composantes_scores['PC2'],
                          c=df_bivarie['Y_Regression'],
                          cmap='coolwarm',
                          alpha=0.6,
                          s=30,
                          vmin=-5, vmax=5)  # Limiter l'échelle à ±5%
    
    ax2.set_xlabel(f'PC1 ({variance_expliquee[0]*100:.1f}%)', fontweight='bold')
    ax2.set_ylabel(f'PC2 ({variance_expliquee[1]*100:.1f}%)', fontweight='bold')
    ax2.set_title('Projection des jours (coloriés par rendement futur)', fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    ax2.axvline(x=0, color='black', linestyle='-', alpha=0.3)
    
    # Colorbar pour les rendements
    cbar2 = plt.colorbar(scatter2, ax=ax2)
    cbar2.set_label('Rendement futur (%)')
else:
    ax2.text(0.5, 0.5, 'Variable Y_Regression non disponible',
             ha='center', va='center', transform=ax2.transAxes, fontsize=12)
    ax2.set_title('Projection non disponible', fontweight='bold')

plt.suptitle('PROJECTION DES INDIVIDUS DANS L\'ESPACE DES COMPOSANTES', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('acp_projection_individus_final.png', dpi=300, bbox_inches='tight')
plt.show()

print(" Graphique sauvegardé : 'acp_projection_individus_final.png'")

# 3. Analyse des corrélations avec la variable cible
print("\n3. CORRELATIONS AVEC LA VARIABLE CIBLE :")

if 'Y_Regression' in df_bivarie.columns:
    print("Corrélation des composantes avec le rendement futur :")
    correlations = []
    for pc in ['PC1', 'PC2', 'PC3']:
        corr = df_bivarie[pc].corr(df_bivarie['Y_Regression'])
        correlations.append((pc, corr))
        print(f"  {pc:5} : ρ = {corr:.4f}")
    
    # Trouver la composante la plus corrélée
    best_pc = max(correlations, key=lambda x: abs(x[1]))
    print(f"\n  Composante la plus prédictive : {best_pc[0]} (|ρ| = {abs(best_pc[1]):.4f})")

# 4. Synthèse finale
print("\n4. SYNTHESE FINALE DE L'ANALYSE MULTIVARIEE :")

print("="*80)
print("RESULTATS PRINCIPAUX DE L'ACP :")
print("="*80)

print(f"\n1. REDUCTION DE DIMENSIONNALITE :")
print(f"   • Variables initiales : {len(variables_corrigees)}")
print(f"   • Composantes pour 80% de variance : {n_composantes}")
print(f"   • Gain en dimension : {(1 - n_composantes/len(variables_corrigees))*100:.0f}%")

print(f"\n2. VARIANCE EXPLIQUEE :")
print(f"   • PC1 : {variance_expliquee[0]*100:.1f}%")
print(f"   • PC2 : {variance_expliquee[1]*100:.1f}%") 
print(f"   • PC3 : {variance_expliquee[2]*100:.1f}%")
print(f"   • Total (3 composantes) : {variance_cumulee[2]*100:.1f}%")

print(f"\n3. INTERPRETATION DES COMPOSANTES :")
print(f"   • PC1 : Dimension de 'NIVEAU DE MARCHE' (prix, volume)")
print(f"   • PC2 : Dimension de 'MOMENTUM & ACTIVITE' (rendements, momentum)")
print(f"   • PC3 : Dimension de 'RISQUE & ACTIVITE ANORMALE' (volatilité, ratio)")

print(f"\n4. IMPLICATIONS POUR LA MODELISATION :")
print(f"   • Les 3 composantes principales peuvent remplacer les 6 variables initiales")
print(f"   • Réduction du risque de multicolinéarité dans les modèles ML")
print(f"   • Variables transformées en dimensions orthogonales")

print(f"\n5. RECOMMANDATIONS :")
print(f"   • Utiliser PC1, PC2, PC3 comme features pour les modèles de prédiction")
print(f"   • Conserver les variables originales pour interprétation")
print(f"   • Standardiser toujours les données avant modélisation")

print("\n" + "="*80)
print(" ETAPE 4 : ANALYSE MULTIVARIEE (ACP) TERMINEE AVEC SUCCES")
print("="*80)

# 5. Préparation pour la phase suivante
print("\n DONNEES PRETES POUR LA PHASE DE MODELISATION ML :")
print(f"DataFrame final shape : {df_bivarie.shape}")
print(f"Colonnes disponibles ({len(df_bivarie.columns)}):")
print(df_bivarie.columns.tolist())


# ========================================
#   ÉTAPE 5 : MODÉLISATION MACHINE LEARNING
# ========================================

### Pour notre projet :
1. Sélection d'au moins 2 algorithmes de classification
2. Analyse de la Matrice de Confusion (faux positifs/négatifs)
3. Comparaison des modèles pour déterminer le plus performant
4. Implications pour le trading algorithmique

### Modèles à implémenter :
- Régression Logistique
- Random Forest  
- SVM (Support Vector Machine)
- K-Nearest Neighbors (KNN)
- XGBoost (option avancée)

In [ ]:
# ========================================
# PREPARATION DES DONNEES POUR LE MACHINE LEARNING
# ========================================

print("ETAPE 5.1 : PREPARATION DES DONNEES POUR LA MODELISATION ML")
print("=" * 80)

# 1. Vérification des données disponibles
print("1. VERIFICATION DES DONNEES DISPONIBLES :")
print(f"DataFrame shape : {df_bivarie.shape}")
print(f"Colonnes disponibles ({len(df_bivarie.columns)}) :")
print(df_bivarie.columns.tolist())

# 2. Sélection des features
print("\n2. SELECTION DES FEATURES :")

# Option A : Utiliser les composantes ACP (recommandé)
features_acp = ['PC1', 'PC2', 'PC3']

# Option B : Variables originales importantes
features_originales = ['Rendement_%', 'Volatilite_20', 'Volume_Log', 
                      'Rendement_5j', 'Volume_Ratio']

print(f"Features sélectionnées (ACP) : {features_acp}")
print(f"  • PC1 : Dimension 'Niveau de marché'")
print(f"  • PC2 : Dimension 'Momentum & Activité'") 
print(f"  • PC3 : Dimension 'Risque & Activité anormale'")

# 3. Sélection de la target
print("\n3. SELECTION DE LA TARGET :")

# Nous avons déjà créé Y_Classification (1 = hausse demain, 0 = baisse/stagnation)
target = 'Y_Classification'

print(f"Target : {target}")
print(f"  • 1 = Hausse demain")
print(f"  • 0 = Baisse/Stagnation demain")
print(f"Distribution :")
print(f"  • Classe 0 : {(df_bivarie[target] == 0).sum()} ({(df_bivarie[target] == 0).mean()*100:.1f}%)")
print(f"  • Classe 1 : {(df_bivarie[target] == 1).sum()} ({(df_bivarie[target] == 1).mean()*100:.1f}%)")

# 4. Création des datasets X et y
print("\n4. CREATION DES DATASETS :")

X = df_bivarie[features_acp].copy()
y = df_bivarie[target].copy()

print(f"Shape des données :")
print(f"  • X (features) : {X.shape}")
print(f"  • y (target) : {y.shape}")

# 5. Split temporel (IMPORTANT pour séries financières)
print("\n5. SPLIT TEMPOREL TRAIN/TEST :")

# Pour les données financières, on ne shuffle pas aléatoirement
# On sépare par date pour éviter le data leakage
split_date = '2023-06-01'  # Environ 70% train, 30% test

X_train = X[X.index < split_date]
X_test = X[X.index >= split_date]
y_train = y[X.index < split_date]
y_test = y[X.index >= split_date]

print(f"Split temporel :")
print(f"  • Date de split : {split_date}")
print(f"  • Train : {X_train.shape[0]} observations ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"    Période : {X_train.index[0].date()} au {X_train.index[-1].date()}")
print(f"  • Test : {X_test.shape[0]} observations ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"    Période : {X_test.index[0].date()} au {X_test.index[-1].date()}")

# 6. Standardisation
print("\n6. STANDARDISATION DES DONNEES :")

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"  • Données standardisées")
print(f"  • Moyenne après standardisation : {X_train_scaled.mean():.6f}")
print(f"  • Écart-type après standardisation : {X_train_scaled.std():.6f}")

# 7. Baseline model
print("\n7. MODELE BASELINE :")

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

# Modèle qui prédit toujours la classe majoritaire
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train_scaled, y_train)
y_pred_baseline = baseline.predict(X_test_scaled)
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)

print(f"  • Stratégie baseline : toujours prédire '{baseline.classes_[0]}'")
print(f"  • Accuracy baseline : {baseline_accuracy:.4f}")
print(f"  • Performance à battre : > {baseline_accuracy:.4f}")

# 8. Vérification finale
print("\n8. VERIFICATION FINALE :")
print(f"✅ Données prêtes pour la modélisation :")
print(f"   • Features : {features_acp}")
print(f"   • Observations train : {X_train_scaled.shape}")
print(f"   • Observations test : {X_test_scaled.shape}")
print(f"   • Baseline accuracy : {baseline_accuracy:.4f}")

In [ ]:
# ========================================
# IMPLEMENTATION ET COMPARAISON DES MODELES
# ========================================

print("ETAPE 5.2 : IMPLEMENTATION DES MODELES DE CLASSIFICATION")
print("=" * 80)

# Importation des modèles et métriques
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                           f1_score, confusion_matrix, classification_report,
                           roc_auc_score, roc_curve)
import warnings
warnings.filterwarnings('ignore')

# 1. Définition des modèles à comparer (sans XGBoost)
print("1. SELECTION DES ALGORITHMES :")

models = {
    'Régression Logistique': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'SVM': SVC(random_state=42, probability=True),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100)
}

print(f"Modèles implémentés ({len(models)}) :")
for name in models.keys():
    print(f"  • {name}")

# 2. Entraînement et évaluation
print("\n2. ENTRAINEMENT ET EVALUATION DES MODELES :")

results = {}

for name, model in models.items():
    print(f"\n--- {name} ---")
    
    # Entraînement
    model.fit(X_train_scaled, y_train)
    
    # Prédictions
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, "predict_proba") else None
    
    # Calcul des métriques
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    
    # ROC-AUC si disponible
    roc_auc = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
    
    # Stockage des résultats
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': roc_auc,
        'confusion_matrix': cm,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    # Affichage des résultats
    print(f"  Accuracy : {accuracy:.4f}")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall : {recall:.4f}")
    print(f"  F1-Score : {f1:.4f}")
    if roc_auc is not None:
        print(f"  ROC-AUC : {roc_auc:.4f}")
    
    # Matrice de confusion
    tn, fp, fn, tp = cm.ravel()
    print(f"  Matrice de confusion :")
    print(f"                 Prédit 0   Prédit 1")
    print(f"  Réel 0         {tn:4d}       {fp:4d}")
    print(f"  Réel 1         {fn:4d}       {tp:4d}")
    
    # Analyse des erreurs
    print(f"  Analyse des erreurs :")
    print(f"    • Faux Positifs (FP) : {fp} → Prédit hausse mais baisse réelle")
    print(f"    • Faux Négatifs (FN) : {fn} → Prédit baisse mais hausse réelle")
    print(f"    • Taux d'erreur : {(fp + fn) / len(y_test):.2%}")

# 3. Comparaison des modèles
print("\n3. COMPARAISON DES PERFORMANCES :")

print("\n" + "="*90)
print("TABLEAU COMPARATIF DES MODELES")
print("="*90)
print(f"{'Modèle':<25} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'ROC-AUC':>10}")
print("-"*90)

for name, metrics in results.items():
    roc_auc_str = f"{metrics['roc_auc']:.4f}" if metrics['roc_auc'] is not None else "N/A"
    print(f"{name:<25} {metrics['accuracy']:>10.4f} {metrics['precision']:>10.4f} "
          f"{metrics['recall']:>10.4f} {metrics['f1_score']:>10.4f} {roc_auc_str:>10}")

print("-"*90)

# 4. Identification du meilleur modèle
print("\n4. IDENTIFICATION DU MEILLEUR MODELE :")

# Par F1-Score (métrique équilibrée)
best_model_name = max(results.items(), key=lambda x: x[1]['f1_score'])[0]
best_model = results[best_model_name]['model']
best_metrics = results[best_model_name]

print(f"• Meilleur modèle par F1-Score : {best_model_name}")
print(f"  F1-Score : {best_metrics['f1_score']:.4f}")
print(f"  Accuracy : {best_metrics['accuracy']:.4f}")
print(f"  Comparé au baseline : {best_metrics['accuracy'] - baseline_accuracy:+.4f} (+{(best_metrics['accuracy'] - baseline_accuracy)/baseline_accuracy*100:.1f}%)")

# 5. Validation croisée sur le meilleur modèle
print("\n5. VALIDATION CROISEE (meilleur modèle) :")

from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(best_model, X_train_scaled, y_train, 
                           cv=5, scoring='f1')

print(f"Validation croisée (5 folds) - {best_model_name} :")
print(f"  Scores F1 : {cv_scores.round(4)}")
print(f"  Moyenne : {cv_scores.mean():.4f} (± {cv_scores.std():.4f})")
print(f"  Performance stable : {'OUI' if cv_scores.std() < 0.05 else 'NON'}")

print("\n✅ IMPLEMENTATION DES MODELES TERMINEE")

In [ ]:
# ========================================
# ANALYSE APPROFONDIE DU MEILLEUR MODÈLE
# ========================================

print("ETAPE 5.3 : ANALYSE APPROFONDIE DU MEILLEUR MODÈLE")
print("=" * 80)

# Récupération du meilleur modèle et de ses prédictions
best_model_name = 'Régression Logistique'  # D'après vos résultats
best_model = models[best_model_name]
best_metrics = results[best_model_name]

y_pred_final = best_metrics['y_pred']
y_pred_proba_final = best_metrics['y_pred_proba']

print(f"Modèle analysé : {best_model_name}")
print(f"Performance : F1-Score = {best_metrics['f1_score']:.4f}, Accuracy = {best_metrics['accuracy']:.4f}")

# 1. Métriques détaillées
print("\n1. METRIQUES DETAILLEES :")

print("Rapport de classification détaillé :")
print(classification_report(y_test, y_pred_final))

# 2. Analyse stratégique pour le trading
print("\n2. ANALYSE STRATEGIQUE POUR LE TRADING :")

# Recalcul de la matrice de confusion
cm_final = confusion_matrix(y_test, y_pred_final)
tn, fp, fn, tp = cm_final.ravel()

print(f"Sur {len(y_test)} jours de test :")
print(f"  • Vrais Positifs (TP) : {tp} → Achat correct, hausse réalisée")
print(f"  • Vrais Négatifs (TN) : {tn} → Pas d'achat correct, baisse évitée")
print(f"  • Faux Positifs (FP) : {fp} → Achat incorrect, PERDRE DE L'ARGENT")
print(f"  • Faux Négatifs (FN) : {fn} → Pas d'achat incorrect, opportunité manquée")

print(f"\nAnalyse risque/rendement :")
print(f"  • Taux de succès des achats (Précision) : {tp/(tp+fp):.1%}")
print(f"    → Seulement {tp/(tp+fp):.1%} des signaux d'achat sont corrects")
print(f"  • Taux de détection des hausses (Recall) : {tp/(tp+fn):.1%}")
print(f"    → Détecte {tp/(tp+fn):.1%} des vraies hausses")
print(f"  • Risque d'erreur d'achat : {fp/(tp+fp):.1%}")
print(f"    → {fp/(tp+fp):.1%} de chance de perdre de l'argent sur un achat")

# 3. Importance des features (coefficients)
print("\n3. IMPORTANCE DES FEATURES (COEFFICIENTS) :")

coefficients = best_model.coef_[0]
feature_coef = pd.DataFrame({
    'Feature': features,
    'Coefficient': coefficients,
    'Impact sur Hausse': ['Favorise' if x > 0 else 'Réduit' for x in coefficients],
    'Importance Absolue': np.abs(coefficients)
}).sort_values('Importance Absolue', ascending=False)

print("Coefficients de la régression logistique :")
print(feature_coef.to_string(index=False))

print("\nInterprétation des coefficients :")
for idx, row in feature_coef.iterrows():
    sign = "+" if row['Coefficient'] > 0 else "-"
    print(f"  • {row['Feature']} : {sign}{abs(row['Coefficient']):.3f}")
    print(f"    → {row['Impact sur Hausse']} la probabilité de hausse")

# 4. Visualisation des coefficients
plt.figure(figsize=(10, 6))
colors = ['red' if x < 0 else 'green' for x in feature_coef['Coefficient']]
plt.barh(feature_coef['Feature'], feature_coef['Coefficient'], color=colors)
plt.xlabel('Coefficient', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Impact des Features sur la Prédiction de Hausse\nRégression Logistique', 
          fontweight='bold', fontsize=14)
plt.axvline(x=0, color='black', linestyle='--', alpha=0.5)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('coefficients_regression_logistique.png', dpi=300, bbox_inches='tight')
plt.show()

# 5. Courbe ROC et choix du seuil optimal
print("\n5. COURBE ROC ET CHOIX DU SEUIL OPTIMAL :")

# Calcul des probabilités
y_pred_proba_final = best_model.predict_proba(X_test_scaled)[:, 1]

# Courbe ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_final)
roc_auc = roc_auc_score(y_test, y_pred_proba_final)

plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, 
         label=f'Courbe ROC (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', 
         label='Performance aléatoire')
plt.xlabel('Taux de Faux Positifs (FP)', fontsize=12)
plt.ylabel('Taux de Vrais Positifs (TP)', fontsize=12)
plt.title('Courbe ROC - Régression Logistique', fontweight='bold', fontsize=14)
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('courbe_roc.png', dpi=300, bbox_inches='tight')
plt.show()

# 6. Recherche du seuil optimal pour le trading
print("\n6. RECHERCHE DU SEUIL OPTIMAL POUR LE TRADING :")

from sklearn.metrics import precision_recall_curve

precision_vals, recall_vals, thresholds_pr = precision_recall_curve(y_test, y_pred_proba_final)

# Trouver le seuil qui maximise F1-Score
f1_scores = 2 * (precision_vals * recall_vals) / (precision_vals + recall_vals + 1e-10)
optimal_idx = np.argmax(f1_scores[:-1])  # Exclure le dernier élément qui a len+1
optimal_threshold = thresholds_pr[optimal_idx]

# Calculer les métriques avec seuil optimal
y_pred_optimal = (y_pred_proba_final >= optimal_threshold).astype(int)
precision_opt = precision_score(y_test, y_pred_optimal, zero_division=0)
recall_opt = recall_score(y_test, y_pred_optimal, zero_division=0)
f1_opt = f1_score(y_test, y_pred_optimal, zero_division=0)

print(f"Seuil par défaut (0.5) :")
print(f"  • Précision : {best_metrics['precision']:.4f}")
print(f"  • Recall : {best_metrics['recall']:.4f}")
print(f"  • F1-Score : {best_metrics['f1_score']:.4f}")

print(f"\nSeuil optimal trouvé ({optimal_threshold:.3f}) :")
print(f"  • Précision : {precision_opt:.4f}")
print(f"  • Recall : {recall_opt:.4f}")
print(f"  • F1-Score : {f1_opt:.4f}")

# Visualisation précision/recall vs seuil
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(thresholds_pr, precision_vals[:-1], 'b-', label='Précision')
plt.plot(thresholds_pr, recall_vals[:-1], 'r-', label='Recall')
plt.axvline(x=optimal_threshold, color='green', linestyle='--', 
           label=f'Seuil optimal ({optimal_threshold:.2f})')
plt.xlabel('Seuil de décision', fontsize=10)
plt.ylabel('Score', fontsize=10)
plt.title('Précision et Recall en fonction du seuil', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(thresholds_pr, f1_scores[:-1], 'g-', label='F1-Score')
plt.axvline(x=optimal_threshold, color='green', linestyle='--', 
           label=f'Seuil optimal ({optimal_threshold:.2f})')
plt.xlabel('Seuil de décision', fontsize=10)
plt.ylabel('F1-Score', fontsize=10)
plt.title('F1-Score en fonction du seuil', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('seuil_optimal_trading.png', dpi=300, bbox_inches='tight')
plt.show()

# 7. Recommandations pour le trading algorithmique
print("\n7. RECOMMANDATIONS POUR LE TRADING ALGORITHMIQUE :")

print(f"\n📈 STRATEGIE AVEC SEUIL PAR DEFAUT (0.5) :")
print(f"   • Nombre de signaux d'achat : {tp+fp}")
print(f"   • Signaux corrects : {tp} ({tp/(tp+fp):.1%})")
print(f"   → Risque élevé : {fp/(tp+fp):.1%} de pertes sur chaque achat")

print(f"\n🎯 STRATEGIE AVEC SEUIL OPTIMAL ({optimal_threshold:.3f}) :")
y_pred_optimal_binary = (y_pred_proba_final >= optimal_threshold).astype(int)
signaux_achat = np.sum(y_pred_optimal_binary == 1)
tp_opt = np.sum((y_pred_optimal_binary == 1) & (y_test == 1))
fp_opt = np.sum((y_pred_optimal_binary == 1) & (y_test == 0))

if signaux_achat > 0:
    taux_succes = tp_opt / signaux_achat
    print(f"   • Nombre de signaux d'achat : {signaux_achat}")
    print(f"   • Signaux corrects : {tp_opt} ({taux_succes:.1%})")
    print(f"   → Qualité améliorée : {taux_succes:.1%} de succès")
else:
    print(f"   • Aucun signal d'achat avec ce seuil strict")

print(f"\n⚠️  CONCLUSION POUR LE BACKTESTING :")
print(f"   1. Performance modeste (F1-Score = {best_metrics['f1_score']:.3f})")
print(f"   2. Nombreux faux positifs ({fp} sur {tp+fp} signaux)")
print(f"   3. Nécessite filtrage supplémentaire ou combinaison avec")
print(f"      d'autres indicateurs techniques")
print(f"   4. Tester avec seuil plus élevé ({optimal_threshold:.3f}) pour")
print(f"      réduire les faux positifs et améliorer la précision")

print("\n✅ ANALYSE DU MEILLEUR MODELE TERMINEE")

In [ ]:
# ========================================
# ÉTAPE 5.5 : STRATÉGIE DE TRADING AMÉLIORÉE
# ========================================

print("ÉTAPE 5.5 : STRATÉGIE DE TRADING OPTIMISÉE POUR BACKTESTING")
print("=" * 80)

# Définir les variables de rendement (hypothèses réalistes)
RENDEMENT_MOYEN_HAUSSE = 0.008   # 0.8% en hausse
RENDEMENT_MOYEN_BAISSE = -0.005  # -0.5% en baisse

print(f"Basé sur l'analyse du meilleur modèle : {best_model_name}")
print(f"Seuil optimal théorique : {optimal_threshold:.3f}")
print(f"Performance avec seuil 0.5 : F1 = {best_metrics['f1_score']:.3f}")

# 1. Problématique identifiée
print("\n1. DIAGNOSTIC DE LA STRATÉGIE :")
print(f"   • Avec seuil {optimal_threshold:.3f} : recall=100% mais trop de trades")
print(f"   • Risque élevé : {fp/(tp+fp):.1%} de faux positifs avec seuil 0.5")
print(f"   • Nécessité d'optimiser pour le trading réel")

# 2. Recherche de seuil équilibré
print("\n2. OPTIMISATION DU SEUIL POUR LE TRADING :")

seuils_a_tester = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75]
resultats_seuils = []

print(f"\n{'Seuil':<8} {'Trades':<10} {'Précision':<12} {'Recall':<10} {'Taux Succès':<12} {'Score':<10}")
print("-" * 70)

for seuil in seuils_a_tester:
    predictions = (y_pred_proba_final >= seuil).astype(int)
    n_trades = np.sum(predictions)
    
    if n_trades > 0:
        precision = precision_score(y_test, predictions, zero_division=0)
        recall = recall_score(y_test, predictions, zero_division=0)
        vrais_positifs = np.sum((predictions == 1) & (y_test.values == 1))
        taux_succes = vrais_positifs / n_trades
        
        # Score qui favorise la précision (critique en trading)
        score = (precision * 0.6 + taux_succes * 0.4) * min(1, n_trades/50)
        
        resultats_seuils.append({
            'seuil': seuil,
            'trades': n_trades,
            'precision': precision,
            'recall': recall,
            'taux_succes': taux_succes,
            'score': score
        })
        
        print(f"{seuil:<8.2f} {n_trades:<10} {precision:<12.3f} {recall:<10.3f} "
              f"{taux_succes:<12.1%} {score:<10.3f}")
    else:
        print(f"{seuil:<8.2f} {'0':<10} {'N/A':<12} {'N/A':<10} {'N/A':<12} {'N/A':<10}")

# Trouver le meilleur seuil pour trading
if resultats_seuils:
    meilleur_seuil = max(resultats_seuils, key=lambda x: x['score'])
    seuil_trading_optime = meilleur_seuil['seuil']
    
    print(f"\n✅ SEUIL OPTIMAL POUR TRADING : {seuil_trading_optime:.2f}")
    print(f"   • Nombre de trades : {meilleur_seuil['trades']}")
    print(f"   • Taux de succès : {meilleur_seuil['taux_succes']:.1%}")
    print(f"   • Précision : {meilleur_seuil['precision']:.3f}")
else:
    seuil_trading_optime = 0.6  # Valeur par défaut
    print(f"\n⚠️  Utilisation du seuil par défaut : {seuil_trading_optime:.2f}")

# 3. Simulation de trading réaliste
print("\n3. SIMULATION DE TRADING AVEC GESTION DU RISQUE :")

# Paramètres réalistes
COMMISSION = 0.001  # 0.1%
STOP_LOSS = -0.015   # -1.5%
TAKE_PROFIT = 0.025  # +2.5%

# Initialisation
capital_initial = 10000
capital = capital_initial
historique_capital = [capital]
n_trades_executes = 0
trades_gagnants = 0
trades_perdants = 0

# Signaux avec seuil optimisé
signaux_trading = (y_pred_proba_final >= seuil_trading_optime).astype(int)

print(f"\nParamètres de simulation :")
print(f"   • Capital initial : {capital_initial} €")
print(f"   • Seuil de décision : {seuil_trading_optime:.2f}")
print(f"   • Commission par trade : {COMMISSION:.1%}")
print(f"   • Stop-loss : {STOP_LOSS:.1%}, Take-profit : {TAKE_PROFIT:.1%}")
print(f"   • Signaux générés : {np.sum(signaux_trading)}")

# Simulation jour par jour
rendements_journaliers = []

for i in range(len(y_test)):
    gain_journalier = 0
    
    if signaux_trading[i] == 1:  # Signal d'achat
        n_trades_executes += 1
        
        if y_test.values[i] == 1:  # Hausse réelle
            gain = min(RENDEMENT_MOYEN_HAUSSE, TAKE_PROFIT)
            trades_gagnants += 1
        else:  # Baisse réelle
            gain = max(RENDEMENT_MOYEN_BAISSE, STOP_LOSS)
            trades_perdants += 1
        
        # Appliquer commission
        gain_net = gain - COMMISSION
        gain_journalier = gain_net
    
    rendements_journaliers.append(gain_journalier)
    capital *= (1 + gain_journalier)
    historique_capital.append(capital)

# Calcul des métriques de performance
rendement_total = capital / capital_initial - 1

# Calcul rendement moyen par trade
rendements_trades = [r for r in rendements_journaliers if r != 0]
rendement_moyen_trade = np.mean(rendements_trades) if rendements_trades else 0

ratio_gain_perte = trades_gagnants / max(trades_perdants, 1)

# Calcul du Sharpe Ratio
rendement_moyen_annuel = np.mean(rendements_journaliers) * 252
volatilite_annuelle = np.std(rendements_journaliers) * np.sqrt(252)
sharpe_ratio = rendement_moyen_annuel / (volatilite_annuelle + 1e-10)

# Calcul du Maximum Drawdown
peak = capital_initial
max_drawdown = 0
for cap in historique_capital:
    if cap > peak:
        peak = cap
    drawdown = (peak - cap) / peak
    if drawdown > max_drawdown:
        max_drawdown = drawdown

print(f"\n📊 RESULTATS DE LA SIMULATION :")
print(f"   • Capital final : {capital:.0f} €")
print(f"   • Rendement total : {rendement_total:+.2%}")
print(f"   • Trades exécutés : {n_trades_executes}")
print(f"   • Trades gagnants : {trades_gagnants} ({trades_gagnants/max(n_trades_executes,1):.1%})")
print(f"   • Ratio gain/pertes : {ratio_gain_perte:.2f}")
print(f"   • Rendement moyen par trade : {rendement_moyen_trade:+.3%}")
print(f"   • Sharpe Ratio : {sharpe_ratio:.3f}")
print(f"   • Maximum Drawdown : {max_drawdown:.2%}")

# 4. Visualisation de la performance
print("\n4. VISUALISATION GRAPHIQUE :")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Graphique 1 : Évolution du capital
axes[0, 0].plot(historique_capital, 'b-', linewidth=2)
axes[0, 0].axhline(y=capital_initial, color='gray', linestyle='--', alpha=0.5)
axes[0, 0].fill_between(range(len(historique_capital)), capital_initial, historique_capital, 
                       where=np.array(historique_capital) >= capital_initial, 
                       alpha=0.3, color='green', label='Gains')
axes[0, 0].set_title(f'Évolution du Capital\nRendement: {rendement_total:+.1%}', 
                     fontweight='bold')
axes[0, 0].set_xlabel('Jours')
axes[0, 0].set_ylabel('Capital (€)')
axes[0, 0].grid(alpha=0.3)
axes[0, 0].legend()

# Graphique 2 : Distribution des rendements des trades
if len(rendements_trades) > 0:
    axes[0, 1].hist(rendements_trades, bins=20, edgecolor='black', alpha=0.7)
    axes[0, 1].axvline(x=0, color='red', linestyle='--', label='Break-even')
    axes[0, 1].set_title(f'Distribution des Rendements\n({len(rendements_trades)} trades)', 
                         fontweight='bold')
    axes[0, 1].set_xlabel('Rendement par trade')
    axes[0, 1].set_ylabel('Fréquence')
    axes[0, 1].grid(alpha=0.3)
    axes[0, 1].legend()
else:
    axes[0, 1].text(0.5, 0.5, 'Aucun trade exécuté', 
                   ha='center', va='center', fontsize=12)
    axes[0, 1].set_title('Distribution des Rendements', fontweight='bold')

# Graphique 3 : Signaux sur 50 derniers jours
jours_a_afficher = min(50, len(y_test))
if jours_a_afficher > 0:
    indices = list(range(-jours_a_afficher, 0))
    
    axes[1, 0].plot(indices, y_test.values[-jours_a_afficher:], 'g-', 
                   alpha=0.5, label='Réalité (Hausse=1)')
    
    # Trouver les indices des signaux
    signaux_indices = []
    signaux_valeurs = []
    for j, idx in enumerate(indices):
        if signaux_trading[-jours_a_afficher:][j] == 1:
            signaux_indices.append(idx)
            signaux_valeurs.append(y_test.values[-jours_a_afficher:][j])
    
    if signaux_indices:
        axes[1, 0].scatter(signaux_indices, signaux_valeurs, c='red', s=50, 
                          marker='^', label='Signal ACHAT', zorder=5)
    
    axes[1, 0].set_title(f'Signaux de Trading\n({jours_a_afficher} derniers jours)', 
                         fontweight='bold')
    axes[1, 0].set_xlabel('Jours (relatifs)')
    axes[1, 0].set_ylabel('Hausse/Baisse')
    axes[1, 0].grid(alpha=0.3)
    axes[1, 0].legend()
else:
    axes[1, 0].text(0.5, 0.5, 'Pas assez de données', 
                   ha='center', va='center', fontsize=12)
    axes[1, 0].set_title('Signaux de Trading', fontweight='bold')

# Graphique 4 : Comparaison des seuils
if resultats_seuils:
    seuils = [r['seuil'] for r in resultats_seuils]
    succes = [r['taux_succes'] for r in resultats_seuils]
    trades = [r['trades'] for r in resultats_seuils]
    
    ax2 = axes[1, 1].twinx()
    line1, = axes[1, 1].plot(seuils, succes, 'b-o', linewidth=2, 
                            label='Taux de succès', markersize=8)
    line2, = ax2.plot(seuils, trades, 'r--s', linewidth=2, 
                     label='Nombre de trades', markersize=8)
    
    # Marquer le seuil optimal
    axes[1, 1].axvline(x=seuil_trading_optime, color='green', 
                      linestyle='--', alpha=0.7, label=f'Seuil optimal')
    
    axes[1, 1].set_xlabel('Seuil de décision')
    axes[1, 1].set_ylabel('Taux de succès', color='b')
    ax2.set_ylabel('Nombre de trades', color='r')
    axes[1, 1].set_title('Optimisation du Seuil de Trading', fontweight='bold')
    axes[1, 1].grid(alpha=0.3)
    
    # Combiner les légendes
    lines = [line1, line2]
    labels = [l.get_label() for l in lines]
    axes[1, 1].legend(lines + [axes[1, 1].lines[-1]], 
                     labels + [f'Seuil optimal ({seuil_trading_optime:.2f})'])
else:
    axes[1, 1].text(0.5, 0.5, 'Pas de résultats de seuils', 
                   ha='center', va='center', fontsize=12)
    axes[1, 1].set_title('Optimisation du Seuil', fontweight='bold')

plt.tight_layout()
plt.savefig('strategie_trading_optimisee.png', dpi=300, bbox_inches='tight')
plt.show()

# 5. Recommandations pour backtesting réel
print("\n5. RECOMMANDATIONS POUR BACKTESTING :")

print(f"\n🎯 STRATÉGIE PROPOSÉE :")
print(f"   • Seuil de trading : {seuil_trading_optime:.2f}")
if resultats_seuils:
    print(f"   • Nombre de trades attendu : ~{meilleur_seuil['trades']} sur {len(y_test)} jours")
    print(f"   • Taux de succès attendu : {meilleur_seuil['taux_succes']:.1%}")



print("\n✅ ÉTAPE 5.5 TERMINÉE - STRATÉGIE DE TRADING OPTIMISÉE")
print("   → Graphique sauvegardé : 'strategie_trading_optimisee.png'")

In [ ]:
# ========================================
# PRÉPARATION DES DONNÉES DE PRIX POUR BACKTESTING
# ========================================

print("🔗 LIEN DES DONNÉES DE PRIX POUR BACKTESTING RÉALISTE")
print("=" * 60)

# Vérifier quelles données sont disponibles
print("DataFrames disponibles :")
dataframes_disponibles = []
for var_name in dir():
    var_value = eval(var_name)
    if isinstance(var_value, pd.DataFrame):
        dataframes_disponibles.append(var_name)
        print(f"  • {var_name}: {var_value.shape}")

# Utiliser df_clean qui contient vos données Apple préparées
if 'df_clean' in dataframes_disponibles:
    df_prices = df_clean[['Close', 'Volume']].copy()
    df_prices['Returns'] = df_prices['Close'].pct_change()
    print(f"\n✅ Données Apple chargées depuis df_clean")
    print(f"   • Période : {df_prices.index[0].date()} au {df_prices.index[-1].date()}")
    print(f"   • Prix Close : {df_prices['Close'].iloc[0]:.2f} → {df_prices['Close'].iloc[-1]:.2f}")
    print(f"   • Variation totale : {(df_prices['Close'].iloc[-1]/df_prices['Close'].iloc[0]-1):.2%}")
elif 'data' in dataframes_disponibles:
    df_prices = data[['Close', 'Volume']].copy()
    df_prices['Returns'] = df_prices['Close'].pct_change()
    print(f"\n✅ Données Apple chargées depuis data")
else:
    print("\n⚠️  DataFrame 'df_clean' ou 'data' non trouvé")
    print("   → Vérifiez vos noms de variables")

# Synchroniser avec les dates de test X_test
print(f"\n📅 SYNCHRONISATION AVEC LES DATES DE TEST :")

# Vérifier que X_test a un index de dates
if hasattr(X_test, 'index'):
    # Filtrer df_prices sur les mêmes dates que X_test
    dates_communes = df_prices.index.intersection(X_test.index)
    
    if len(dates_communes) > 0:
        df_prices_test = df_prices.loc[dates_communes].copy()
        print(f"   • Dates correspondantes trouvées : {len(dates_communes)}/{len(X_test)}")
        print(f"   • Période backtesting : {df_prices_test.index[0].date()} au {df_prices_test.index[-1].date()}")
        
        # Réaligner y_test avec ces dates
        y_test_aligned = y_test.loc[dates_communes]
        X_test_aligned = X_test.loc[dates_communes]
        y_pred_proba_aligned = y_pred_proba_final[pd.Series(X_test.index).isin(dates_communes)]
        
        print(f"   • Données réalignées avec succès")
    else:
        print("⚠️  Aucune date commune entre df_prices et X_test")
        print("   → Vérifiez les périodes temporelles")
else:
    print("⚠️  X_test n'a pas d'index de dates")
    df_prices_test = df_prices.iloc[-len(X_test):].copy()  # Prendre les N dernières lignes
    print(f"   • Utilisation des {len(df_prices_test)} dernières lignes de df_prices")

print("\n📊 CARACTÉRISTIQUES DES DONNÉES DE PRIX :")
print(f"   • Prix moyen : {df_prices_test['Close'].mean():.2f} $")
print(f"   • Volatilité (std) : {df_prices_test['Returns'].std()*100:.2f}%")
print(f"   • Rendement moyen journalier : {df_prices_test['Returns'].mean()*100:.3f}%")
print(f"   • Skewness : {df_prices_test['Returns'].skew():.3f}")
print(f"   • Kurtosis : {df_prices_test['Returns'].kurtosis():.3f}")

# Calculer le rendement Buy & Hold pour la période
rendement_bh_reel = (df_prices_test['Close'].iloc[-1] / df_prices_test['Close'].iloc[0] - 1)
print(f"   • Rendement Buy & Hold réel : {rendement_bh_reel:+.2%}")

# Visualisation rapide
plt.figure(figsize=(12, 4))
plt.plot(df_prices_test.index, df_prices_test['Close'], 'b-', linewidth=1.5, alpha=0.7)
plt.title(f'Prix Apple (AAPL) - Période de Backtesting\n'
          f'{df_prices_test.index[0].date()} au {df_prices_test.index[-1].date()}', 
          fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Prix ($)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('prix_apple_backtesting.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ DONNÉES DE PRIX PRÊTES POUR BACKTESTING RÉALISTE")

In [ ]:
# ========================================
# ÉTAPE 6 : BACKTESTING DE LA STRATÉGIE ML AVEC DONNÉES RÉELLES
# ========================================

print("ÉTAPE 6 : BACKTESTING COMPLET AVEC DONNÉES APPLE RÉELLES")
print("=" * 80)

print("""
OBJECTIFS DU BACKTESTING :
1. Évaluer la performance réelle de la stratégie sur Apple
2. Calculer les métriques de risque/rendement avec prix réels
3. Tester la robustesse sur différentes périodes
4. Comparer avec la performance réelle de Buy & Hold Apple
5. Valider la viabilité pour le trading en temps réel
""")

# ========================================
# 0. PRÉPARATION DES DONNÉES DE PRIX APPLE
# ========================================

print("\n0. CHARGEMENT DES DONNÉES DE PRIX APPLE POUR BACKTESTING")
print("-" * 60)

# Vérifier quelles données sont disponibles
try:
    # Essayer d'abord avec df_clean (données préparées)
    df_prices = df_clean[['Close', 'Volume']].copy()
    print(f"✅ Données chargées depuis df_clean")
except NameError:
    try:
        # Sinon essayer avec data (données brutes)
        df_prices = data[['Close', 'Volume']].copy()
        print(f"✅ Données chargées depuis data")
    except NameError:
        print("⚠️  Données Apple non trouvées")
        print("   → Vérifiez que vous avez exécuté les cellules de téléchargement")
        raise

# Calculer les rendements
df_prices['Returns'] = df_prices['Close'].pct_change()

print(f"   • Période totale : {df_prices.index[0].date()} au {df_prices.index[-1].date()}")
print(f"   • Nombre de jours : {len(df_prices)}")
print(f"   • Prix initial : {df_prices['Close'].iloc[0]:.2f} $")
print(f"   • Prix final : {df_prices['Close'].iloc[-1]:.2f} $")
print(f"   • Rendement total Apple : {(df_prices['Close'].iloc[-1]/df_prices['Close'].iloc[0]-1):.2%}")

# Synchroniser avec les dates de test X_test
print(f"\n📅 SYNCHRONISATION AVEC LES DATES DE TEST ML :")

if hasattr(X_test, 'index') and hasattr(X_test.index[0], 'date'):
    # Filtrer df_prices sur les mêmes dates que X_test
    dates_communes = df_prices.index.intersection(X_test.index)
    
    if len(dates_communes) > 0:
        df_prices_test = df_prices.loc[dates_communes].copy()
        
        # Réaligner les prédictions avec ces dates
        mask = pd.Series(X_test.index).isin(dates_communes)
        y_test_aligned = y_test.loc[dates_communes]
        X_test_aligned = X_test.loc[dates_communes]
        
        # Pour y_pred_proba_final, on doit garder le même ordre
        if 'y_pred_proba_final' in globals():
            # Créer un DataFrame temporaire pour l'alignement
            temp_df = pd.DataFrame({
                'index': X_test.index,
                'proba': y_pred_proba_final
            })
            temp_df = temp_df[temp_df['index'].isin(dates_communes)]
            y_pred_proba_aligned = temp_df['proba'].values
        
        print(f"   ✅ Dates correspondantes : {len(dates_communes)}/{len(X_test)} jours")
        print(f"   • Période backtesting : {df_prices_test.index[0].date()} au {df_prices_test.index[-1].date()}")
        
    else:
        print("⚠️  Aucune date commune trouvée")
        print("   → Utilisation des dernières données disponibles")
        df_prices_test = df_prices.iloc[-len(X_test):].copy()
else:
    print("⚠️  X_test sans index de dates")
    print("   → Utilisation des dernières données disponibles")
    df_prices_test = df_prices.iloc[-len(X_test):].copy()

# Calculer le rendement Buy & Hold réel pour la période
rendement_bh_reel = (df_prices_test['Close'].iloc[-1] / df_prices_test['Close'].iloc[0] - 1)

print(f"\n📊 CARACTÉRISTIQUES DE LA PÉRIODE DE TEST :")
print(f"   • Prix de départ : {df_prices_test['Close'].iloc[0]:.2f} $")
print(f"   • Prix de fin : {df_prices_test['Close'].iloc[-1]:.2f} $")
print(f"   • Rendement Buy & Hold réel : {rendement_bh_reel:+.2%}")
print(f"   • Volatilité (std) : {df_prices_test['Returns'].std()*100:.2f}%")
print(f"   • Ratio de Sharpe brut : {(df_prices_test['Returns'].mean()/df_prices_test['Returns'].std())*np.sqrt(252):.3f}")

# ========================================
# 1. IMPLÉMENTATION DU MOTEUR DE BACKTESTING AMÉLIORÉ
# ========================================

print("\n1. IMPLÉMENTATION DU MOTEUR DE BACKTESTING POUR APPLE")
print("-" * 60)

class BacktesterApple:
    """Classe spécialisée pour le backtesting sur Apple"""
    
    def __init__(self, capital_initial=10000, commission=0.001, slippage=0.0005):
        self.capital_initial = capital_initial
        self.commission = commission
        self.slippage = slippage  # Impact de marché
        self.capital = capital_initial
        self.historique_capital = [capital_initial]
        self.historique_rendements = []
        self.trades = []
        self.position = 0  # 0 = neutre, 1 = long
        self.entry_price = 0
        
    def execute_trade(self, signal, current_price, date, stop_loss=None, take_profit=None):
        """Exécute un trade avec gestion de risque"""
        
        rendement_jour = 0
        trade_closed = False
        
        # Gestion de la position existante
        if self.position == 1:
            # Calculer le rendement depuis l'entrée
            price_change = (current_price - self.entry_price) / self.entry_price
            
            # Vérifier stop-loss et take-profit
            close_position = False
            if stop_loss is not None and price_change <= stop_loss:
                price_change = stop_loss
                close_position = True
                reason = "STOP_LOSS"
            elif take_profit is not None and price_change >= take_profit:
                price_change = take_profit
                close_position = True
                reason = "TAKE_PROFIT"
            elif signal == 0:  # Signal de sortie
                close_position = True
                reason = "SIGNAL_SORTIE"
            
            if close_position:
                # Fermer la position
                rendement = price_change - self.commission - self.slippage
                self.position = 0
                trade_closed = True
                
                # Mettre à jour le dernier trade
                if self.trades:
                    self.trades[-1]['exit_date'] = date
                    self.trades[-1]['exit_price'] = current_price
                    self.trades[-1]['return'] = rendement
                    self.trades[-1]['exit_reason'] = reason
                    self.trades[-1]['duration'] = (date - self.trades[-1]['entry_date']).days
                
                rendement_jour = rendement
        
        # Ouvrir une nouvelle position
        if signal == 1 and self.position == 0:
            self.position = 1
            self.entry_price = current_price * (1 + self.slippage)  # Slippage à l'entrée
            
            self.trades.append({
                'entry_date': date,
                'entry_price': self.entry_price,
                'signal': 1,
                'type': 'LONG'
            })
        
        return rendement_jour, trade_closed
    
    def run_strategy(self, signaux, df_prices, stop_loss=-0.02, take_profit=0.03):
        """Exécute la stratégie complète"""
        
        capital = self.capital_initial
        self.historique_capital = [capital]
        
        # S'assurer que les longueurs correspondent
        min_len = min(len(signaux), len(df_prices))
        signaux = signaux[:min_len]
        prices = df_prices['Close'].values[:min_len]
        dates = df_prices.index[:min_len]
        
        for i in range(min_len):
            rendement, _ = self.execute_trade(
                signal=signaux[i],
                current_price=prices[i],
                date=dates[i],
                stop_loss=stop_loss,
                take_profit=take_profit
            )
            
            # Mettre à jour le capital
            capital *= (1 + rendement)
            self.historique_capital.append(capital)
            self.historique_rendements.append(rendement)
        
        self.capital = capital
        return capital
    
    def calculate_metrics(self):
        """Calcule les métriques de performance détaillées"""
        
        if len(self.historique_rendements) == 0:
            return {}
        
        rendements = np.array(self.historique_rendements)
        
        # Métriques de base
        rendement_total = (self.capital / self.capital_initial) - 1
        rendement_moyen = np.mean(rendements)
        volatilite = np.std(rendements)
        
        # Ratio de Sharpe
        sharpe = (rendement_moyen / (volatilite + 1e-10)) * np.sqrt(252)
        
        # Maximum Drawdown
        peaks = np.maximum.accumulate(self.historique_capital)
        drawdowns = (peaks - self.historique_capital) / peaks
        max_drawdown = np.max(drawdowns) if len(drawdowns) > 0 else 0
        
        # Métriques sur les trades
        trades_completed = [t for t in self.trades if 'return' in t]
        if trades_completed:
            returns_trades = [t['return'] for t in trades_completed]
            wins = [r for r in returns_trades if r > 0]
            losses = [r for r in returns_trades if r < 0]
            
            win_rate = len(wins) / len(returns_trades)
            avg_win = np.mean(wins) if wins else 0
            avg_loss = np.mean(losses) if losses else 0
            profit_factor = abs(sum(wins)) / abs(sum(losses)) if losses else float('inf')
            
            # Ratio de gain/pertes
            win_loss_ratio = abs(avg_win / avg_loss) if avg_loss != 0 else float('inf')
            
            # Expectancy
            expectancy = (win_rate * avg_win) - ((1 - win_rate) * abs(avg_loss))
        else:
            win_rate = avg_win = avg_loss = profit_factor = win_loss_ratio = expectancy = 0
        
        # Calculs avancés
        positive_returns = rendements[rendements > 0]
        negative_returns = rendements[rendements < 0]
        
        sortino_ratio = (rendement_moyen / (np.std(negative_returns) + 1e-10)) * np.sqrt(252)
        calmar_ratio = rendement_total / (max_drawdown + 1e-10)
        
        return {
            'capital_final': self.capital,
            'rendement_total': rendement_total,
            'rendement_moyen_journalier': rendement_moyen,
            'volatilite_journaliere': volatilite,
            'sharpe_ratio': sharpe,
            'sortino_ratio': sortino_ratio,
            'calmar_ratio': calmar_ratio,
            'max_drawdown': max_drawdown,
            'nombre_trades': len(trades_completed),
            'win_rate': win_rate,
            'avg_win': avg_win,
            'avg_loss': avg_loss,
            'profit_factor': profit_factor,
            'win_loss_ratio': win_loss_ratio,
            'expectancy': expectancy,
            'total_wins': len(wins) if 'wins' in locals() else 0,
            'total_losses': len(losses) if 'losses' in locals() else 0
        }
    
    def generate_report(self):
        """Génère un rapport détaillé"""
        
        metrics = self.calculate_metrics()
        
        report = f"""
        RAPPORT DE BACKTESTING - STRATÉGIE ML APPLE
        {'='*50}
        
        PERFORMANCE GLOBALE :
        • Capital initial : {self.capital_initial:,.0f} $
        • Capital final : {metrics['capital_final']:,.0f} $
        • Rendement total : {metrics['rendement_total']:+.2%}
        • Rendement annualisé : {(1+metrics['rendement_total'])**(252/len(self.historique_rendements))-1:+.2%}
        
        MÉTRIQUES DE RISQUE :
        • Volatilité journalière : {metrics['volatilite_journaliere']:.3%}
        • Sharpe Ratio : {metrics['sharpe_ratio']:.3f}
        • Sortino Ratio : {metrics['sortino_ratio']:.3f}
        • Max Drawdown : {metrics['max_drawdown']:.2%}
        • Calmar Ratio : {metrics['calmar_ratio']:.3f}
        
        PERFORMANCE DES TRADES :
        • Nombre de trades : {metrics['nombre_trades']}
        • Taux de succès (Win Rate) : {metrics['win_rate']:.1%}
        • Gain moyen : {metrics['avg_win']:+.3%}
        • Perte moyenne : {metrics['avg_loss']:+.3%}
        • Profit Factor : {metrics['profit_factor']:.2f}
        • Ratio Gain/Pertes : {metrics['win_loss_ratio']:.2f}
        • Expectancy : {metrics['expectancy']:+.4%}
        • Trades gagnants : {metrics['total_wins']}
        • Trades perdants : {metrics['total_losses']}
        
        STATISTIQUES :
        • Période : {len(self.historique_rendements)} jours
        • Rendement moyen journalier : {metrics['rendement_moyen_journalier']:.4%}
        • Meilleur jour : {np.max(self.historique_rendements):+.3%}
        • Pire jour : {np.min(self.historique_rendements):+.3%}
        """
        
        return report

# ========================================
# 2. EXÉCUTION DU BACKTESTING
# ========================================

print("\n2. EXÉCUTION DU BACKTESTING SUR APPLE")
print("-" * 60)

# Initialiser le backtester
backtester = BacktesterApple(
    capital_initial=10000,  # 10,000 $
    commission=0.001,       # 0.1%
    slippage=0.0005         # 0.05% slippage
)

# Utiliser les signaux ML (seuil optimal de l'étape 5.5)
if 'seuil_trading_optime' in globals():
    seuil = seuil_trading_optime
else:
    seuil = 0.5  # Valeur par défaut

signaux_backtest = (y_pred_proba_final >= seuil).astype(int)

print(f"PARAMÈTRES DU BACKTEST :")
print(f"   • Capital initial : {backtester.capital_initial:,.0f} $")
print(f"   • Commission par trade : {backtester.commission:.1%}")
print(f"   • Slippage : {backtester.slippage:.1%}")
print(f"   • Seuil de décision ML : {seuil:.3f}")
print(f"   • Stop-loss : -2.0%, Take-profit : +3.0%")
print(f"   • Période backtesting : {len(df_prices_test)} jours")
print(f"   • Signaux ML générés : {np.sum(signaux_backtest)}/{len(signaux_backtest)}")

# Ajuster les longueurs
min_len = min(len(signaux_backtest), len(df_prices_test))
signaux_backtest = signaux_backtest[:min_len]
df_prices_test_aligned = df_prices_test.iloc[:min_len]

print(f"\n📊 RENDUREMENTS DE BASE APPLE (PÉRIODE DE TEST) :")
print(f"   • Buy & Hold Apple : {rendement_bh_reel:+.2%}")
print(f"   • Volatilité Apple : {df_prices_test['Returns'].std()*100:.2f}%")
print(f"   • Ratio de Sharpe Apple : {(df_prices_test['Returns'].mean()/df_prices_test['Returns'].std())*np.sqrt(252):.3f}")

# Exécuter la stratégie
print(f"\n⚡ EXÉCUTION DE LA STRATÉGIE ML...")
capital_final = backtester.run_strategy(
    signaux=signaux_backtest,
    df_prices=df_prices_test_aligned,
    stop_loss=-0.02,
    take_profit=0.03
)

# Calculer les métriques
metriques = backtester.calculate_metrics()

print(f"\n🎯 RÉSULTATS DE LA STRATÉGIE ML :")
print(f"   • Capital final : {metriques['capital_final']:,.0f} $")
print(f"   • Rendement total : {metriques['rendement_total']:+.2%}")
print(f"   • Alpha vs Buy&Hold : {metriques['rendement_total'] - rendement_bh_reel:+.2%}")
print(f"   • Sharpe Ratio : {metriques['sharpe_ratio']:.3f}")
print(f"   • Max Drawdown : {metriques['max_drawdown']:.2%}")
print(f"   • Win Rate : {metriques['win_rate']:.1%}")
print(f"   • Profit Factor : {metriques['profit_factor']:.2f}")

# ========================================
# 3. COMPARAISON AVEC BENCHMARKS
# ========================================

print("\n3. COMPARAISON AVEC DIFFÉRENTES STRATÉGIES")
print("-" * 60)

# Stratégie 2: Momentum simple (benchmark technique)
signaux_momentum = (df_prices_test['Returns'].shift(1) > 0).astype(int).fillna(0).values[:min_len]

backtester_momentum = BacktesterApple(capital_initial=10000)
capital_momentum = backtester_momentum.run_strategy(
    signaux=signaux_momentum,
    df_prices=df_prices_test_aligned,
    stop_loss=-0.02,
    take_profit=0.03
)
metriques_momentum = backtester_momentum.calculate_metrics()

# Stratégie 3: Contrarian (inverse du momentum)
signaux_contra = (df_prices_test['Returns'].shift(1) < 0).astype(int).fillna(0).values[:min_len]

backtester_contra = BacktesterApple(capital_initial=10000)
capital_contra = backtester_contra.run_strategy(
    signaux=signaux_contra,
    df_prices=df_prices_test_aligned,
    stop_loss=-0.02,
    take_profit=0.03
)
metriques_contra = backtester_contra.calculate_metrics()

# Stratégie 4: Aléatoire
np.random.seed(42)
signaux_random = np.random.choice([0, 1], size=min_len, p=[0.7, 0.3])  # 30% de temps en position

backtester_random = BacktesterApple(capital_initial=10000)
capital_random = backtester_random.run_strategy(
    signaux=signaux_random,
    df_prices=df_prices_test_aligned,
    stop_loss=-0.02,
    take_profit=0.03
)
metriques_random = backtester_random.calculate_metrics()

print(f"{'STRATÉGIE':<20} {'RENDEMENT':<12} {'SHARPE':<10} {'WIN RATE':<12} {'MAX DD':<10}")
print("-" * 70)
print(f"{'Buy & Hold':<20} {rendement_bh_reel:<+12.2%} {'N/A':<10} {'N/A':<12} {'N/A':<10}")
print(f"{'ML (notre)':<20} {metriques['rendement_total']:<+12.2%} {metriques['sharpe_ratio']:<10.3f} {metriques['win_rate']:<12.1%} {metriques['max_drawdown']:<10.2%}")
print(f"{'Momentum':<20} {metriques_momentum['rendement_total']:<+12.2%} {metriques_momentum['sharpe_ratio']:<10.3f} {metriques_momentum['win_rate']:<12.1%} {metriques_momentum['max_drawdown']:<10.2%}")
print(f"{'Contrarian':<20} {metriques_contra['rendement_total']:<+12.2%} {metriques_contra['sharpe_ratio']:<10.3f} {metriques_contra['win_rate']:<12.1%} {metriques_contra['max_drawdown']:<10.2%}")
print(f"{'Aléatoire':<20} {metriques_random['rendement_total']:<+12.2%} {metriques_random['sharpe_ratio']:<10.3f} {metriques_random['win_rate']:<12.1%} {metriques_random['max_drawdown']:<10.2%}")

# ========================================
# 4. ANALYSE DE ROBUSTESSE
# ========================================

print("\n4. ANALYSE DE ROBUSTESSE (WALK-FORWARD)")
print("-" * 60)

# Diviser en 4 fenêtres temporelles
n_fenetres = 4
fenetre_size = min_len // n_fenetres

print(f"Analyse walk-forward sur {n_fenetres} fenêtres de {fenetre_size} jours :")
print("\n" + "="*80)

resultats_fenetres = []

for i in range(n_fenetres):
    debut = i * fenetre_size
    fin = debut + fenetre_size if i < n_fenetres - 1 else min_len
    
    # Extraire la fenêtre
    signaux_fenetre = signaux_backtest[debut:fin]
    df_fenetre = df_prices_test_aligned.iloc[debut:fin]
    
    # Backtester sur la fenêtre
    bt_fenetre = BacktesterApple(capital_initial=10000)
    capital_fenetre = bt_fenetre.run_strategy(
        signaux=signaux_fenetre,
        df_prices=df_fenetre,
        stop_loss=-0.02,
        take_profit=0.03
    )
    metriques_fenetre = bt_fenetre.calculate_metrics()
    
    # Calculer Buy & Hold pour la fenêtre
    bh_fenetre = (df_fenetre['Close'].iloc[-1] / df_fenetre['Close'].iloc[0] - 1)
    
    resultats_fenetres.append({
        'fenetre': i+1,
        'periode': f"{df_fenetre.index[0].date()} au {df_fenetre.index[-1].date()}",
        'rendement_ml': metriques_fenetre['rendement_total'],
        'rendement_bh': bh_fenetre,
        'sharpe': metriques_fenetre['sharpe_ratio'],
        'win_rate': metriques_fenetre['win_rate'],
        'trades': metriques_fenetre['nombre_trades']
    })
    
    print(f"FENÊTRE {i+1}: {df_fenetre.index[0].date()} au {df_fenetre.index[-1].date()}")
    print(f"   • ML : {metriques_fenetre['rendement_total']:+.2%} (Sharpe: {metriques_fenetre['sharpe_ratio']:.3f}, Win: {metriques_fenetre['win_rate']:.1%})")
    print(f"   • B&H: {bh_fenetre:+.2%}")
    print(f"   • Alpha: {metriques_fenetre['rendement_total'] - bh_fenetre:+.2%}")
    print(f"   • Trades: {metriques_fenetre['nombre_trades']}")
    print()

# Calculer les statistiques de robustesse
rendements_ml = [r['rendement_ml'] for r in resultats_fenetres]
rendements_bh = [r['rendement_bh'] for r in resultats_fenetres]
alphas = [r['rendement_ml'] - r['rendement_bh'] for r in resultats_fenetres]

print("📊 STATISTIQUES DE ROBUSTESSE :")
print(f"   • Rendement ML moyen par fenêtre : {np.mean(rendements_ml):+.2%} (±{np.std(rendements_ml):.2%})")
print(f"   • Rendement B&H moyen : {np.mean(rendements_bh):+.2%} (±{np.std(rendements_bh):.2%})")
print(f"   • Alpha moyen : {np.mean(alphas):+.2%}")
print(f"   • Consistance (écart-type ML) : {'ÉLEVÉE' if np.std(rendements_ml) < 0.05 else 'MOYENNE' if np.std(rendements_ml) < 0.1 else 'FAIBLE'}")
print(f"   • Taux de succès fenêtres positives : {sum(1 for r in rendements_ml if r > 0)}/{n_fenetres}")

# ========================================
# 5. VISUALISATION DES RÉSULTATS
# ========================================

print("\n5. VISUALISATION GRAPHIQUE DES RÉSULTATS")
print("-" * 60)

fig = plt.figure(figsize=(18, 12))

# Graphique 1: Évolution des capitaux
ax1 = plt.subplot(3, 3, 1)
# Capital ML
ax1.plot(backtester.historique_capital, 'b-', linewidth=2, label=f'ML Strategy ({metriques["rendement_total"]:+.1%})')
# Buy & Hold
capital_bh = 10000 * (1 + rendement_bh_reel * np.linspace(0, 1, len(backtester.historique_capital)))
ax1.plot(capital_bh, 'g--', linewidth=2, label=f'Buy & Hold ({rendement_bh_reel:+.1%})')
# Ligne de référence
ax1.axhline(y=10000, color='gray', linestyle=':', alpha=0.5)
ax1.set_title('Évolution du Capital', fontweight='bold', fontsize=14)
ax1.set_xlabel('Jours')
ax1.set_ylabel('Capital ($)')
ax1.legend()
ax1.grid(alpha=0.3)

# Graphique 2: Drawdown
ax2 = plt.subplot(3, 3, 2)
peaks = np.maximum.accumulate(backtester.historique_capital)
drawdowns = (peaks - backtester.historique_capital) / peaks
ax2.fill_between(range(len(drawdowns)), 0, drawdowns, color='red', alpha=0.3)
ax2.plot(drawdowns, 'r-', linewidth=1, alpha=0.7)
ax2.set_title(f'Drawdown (Max = {metriques["max_drawdown"]:.2%})', fontweight='bold', fontsize=14)
ax2.set_xlabel('Jours')
ax2.set_ylabel('Drawdown')
ax2.grid(alpha=0.3)
ax2.set_ylim(0, max(drawdowns) * 1.1)

# Graphique 3: Distribution des rendements des trades
ax3 = plt.subplot(3, 3, 3)
if backtester.trades and 'return' in backtester.trades[0]:
    returns_trades = [t['return'] for t in backtester.trades if 'return' in t]
    if returns_trades:
        ax3.hist(returns_trades, bins=20, edgecolor='black', alpha=0.7, color='skyblue')
        ax3.axvline(x=0, color='red', linestyle='--', alpha=0.7, label='Break-even')
        ax3.axvline(x=np.mean(returns_trades), color='green', linestyle='-', alpha=0.7, label=f'Moyenne: {np.mean(returns_trades):+.3%}')
        ax3.set_title(f'Distribution des Rendements\n({len(returns_trades)} trades)', fontweight='bold', fontsize=14)
        ax3.set_xlabel('Rendement par trade')
        ax3.set_ylabel('Fréquence')
        ax3.legend()
        ax3.grid(alpha=0.3)

# Graphique 4: Signaux vs Prix
ax4 = plt.subplot(3, 3, 4)
# Prix
ax4.plot(df_prices_test_aligned.index, df_prices_test_aligned['Close'], 'k-', alpha=0.7, label='Prix Apple')
ax4.set_ylabel('Prix ($)', color='black')
ax4.tick_params(axis='y', labelcolor='black')
ax4.set_title('Signaux de Trading sur Prix', fontweight='bold', fontsize=14)

# Signaux d'achat
ax4_twin = ax4.twinx()
buy_signals = signaux_backtest == 1
buy_dates = df_prices_test_aligned.index[buy_signals]
buy_prices = df_prices_test_aligned['Close'].values[buy_signals]
ax4_twin.scatter(buy_dates, np.ones(len(buy_dates)), color='green', 
                marker='^', s=50, alpha=0.6, label='Signal ACHAT')
ax4_twin.set_ylabel('Signaux', color='green')
ax4_twin.tick_params(axis='y', labelcolor='green')
ax4_twin.set_ylim(0.5, 1.5)

# Graphique 5: Performance par fenêtre
ax5 = plt.subplot(3, 3, 5)
fenetres = [f'F{i+1}' for i in range(n_fenetres)]
rendements_ml_f = [r['rendement_ml'] for r in resultats_fenetres]
rendements_bh_f = [r['rendement_bh'] for r in resultats_fenetres]

x = np.arange(len(fenetres))
width = 0.35
ax5.bar(x - width/2, rendements_ml_f, width, label='ML Strategy', color='blue', alpha=0.7)
ax5.bar(x + width/2, rendements_bh_f, width, label='Buy & Hold', color='green', alpha=0.7)
ax5.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax5.set_xlabel('Fenêtre')
ax5.set_ylabel('Rendement')
ax5.set_title('Performance par Fenêtre', fontweight='bold', fontsize=14)
ax5.set_xticks(x)
ax5.set_xticklabels(fenetres)
ax5.legend()
ax5.grid(axis='y', alpha=0.3)

# Graphique 6: Métriques comparatives
ax6 = plt.subplot(3, 3, 6)
strategies = ['ML', 'Momentum', 'Contrarian', 'Aléatoire']
sharpe_values = [metriques['sharpe_ratio'], metriques_momentum['sharpe_ratio'], 
                 metriques_contra['sharpe_ratio'], metriques_random['sharpe_ratio']]
win_rates = [metriques['win_rate'], metriques_momentum['win_rate'], 
             metriques_contra['win_rate'], metriques_random['win_rate']]

x = np.arange(len(strategies))
ax6.bar(x - 0.2, sharpe_values, 0.4, label='Sharpe Ratio', color='orange', alpha=0.7)
ax6.bar(x + 0.2, win_rates, 0.4, label='Win Rate', color='purple', alpha=0.7)
ax6.set_xlabel('Stratégie')
ax6.set_ylabel('Valeur')
ax6.set_title('Comparaison des Stratégies', fontweight='bold', fontsize=14)
ax6.set_xticks(x)
ax6.set_xticklabels(strategies)
ax6.legend()
ax6.grid(axis='y', alpha=0.3)

# Graphique 7: Rolling Sharpe Ratio (6 mois)
ax7 = plt.subplot(3, 3, 7)
window = 126  # 6 mois
if len(backtester.historique_rendements) > window:
    rolling_returns = pd.Series(backtester.historique_rendements).rolling(window=window)
    rolling_sharpe = (rolling_returns.mean() / rolling_returns.std()) * np.sqrt(252)
    ax7.plot(rolling_sharpe, 'b-', linewidth=1.5)
    ax7.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    ax7.axhline(y=1, color='green', linestyle=':', alpha=0.5, label='Sharpe=1')
    ax7.set_title(f'Sharpe Ratio Mobile ({window} jours)', fontweight='bold', fontsize=14)
    ax7.set_xlabel('Jours')
    ax7.set_ylabel('Sharpe Ratio')
    ax7.legend()
    ax7.grid(alpha=0.3)

# Graphique 8: Heatmap des performances mensuelles
ax8 = plt.subplot(3, 3, 8)
try:
    # Créer un DataFrame avec rendements par mois
    df_results = pd.DataFrame({
        'date': df_prices_test_aligned.index,
        'return': backtester.historique_rendements
    })
    df_results['year'] = df_results['date'].dt.year
    df_results['month'] = df_results['date'].dt.month
    
    # Calculer rendements mensuels
    monthly_returns = df_results.groupby(['year', 'month'])['return'].apply(lambda x: np.prod(1 + x) - 1)
    monthly_returns = monthly_returns.unstack(level=0)
    
    # Heatmap
    im = ax8.imshow(monthly_returns.values.T, cmap='RdYlGn', aspect='auto', 
                   vmin=-0.1, vmax=0.1)
    ax8.set_title('Rendements Mensuels (Heatmap)', fontweight='bold', fontsize=14)
    ax8.set_xlabel('Mois')
    ax8.set_ylabel('Année')
    ax8.set_xticks(range(12))
    ax8.set_xticklabels(['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Jun', 
                        'Jul', 'Aoû', 'Sep', 'Oct', 'Nov', 'Déc'])
    
    # Ajouter valeurs dans les cellules
    for i in range(monthly_returns.shape[0]):
        for j in range(monthly_returns.shape[1]):
            if not np.isnan(monthly_returns.iloc[i, j]):
                text = ax8.text(i, j, f'{monthly_returns.iloc[i, j]:.1%}', 
                              ha='center', va='center', 
                              color='black' if abs(monthly_returns.iloc[i, j]) < 0.05 else 'white',
                              fontsize=8)
    
    plt.colorbar(im, ax=ax8, label='Rendement')
except:
    ax8.text(0.5, 0.5, 'Données insuffisantes\npour heatmap', 
            ha='center', va='center', fontsize=12)
    ax8.set_title('Rendements Mensuels', fontweight='bold', fontsize=14)

# Graphique 9: Équity Curve vs Drawdown
ax9 = plt.subplot(3, 3, 9)
# Normaliser l'équity curve
equity_normalized = np.array(backtester.historique_capital) / 10000
ax9.plot(equity_normalized, 'b-', linewidth=2, label='Equity Curve')
ax9.fill_between(range(len(drawdowns)), equity_normalized - drawdowns, equity_normalized, 
                color='red', alpha=0.3, label='Drawdown')
ax9.axhline(y=1, color='gray', linestyle=':', alpha=0.5, label='Capital initial')
ax9.set_title('Equity Curve et Drawdown', fontweight='bold', fontsize=14)
ax9.set_xlabel('Jours')
ax9.set_ylabel('Multiplicateur de Capital')
ax9.legend()
ax9.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('backtesting_apple_complet.png', dpi=300, bbox_inches='tight')
plt.show()

# ========================================
# 6. RAPPORT FINAL ET RECOMMANDATIONS
# ========================================

print("\n6. RAPPORT FINAL ET RECOMMANDATIONS")
print("=" * 60)

# Générer le rapport
rapport = backtester.generate_report()
print(rapport)

print("\n📊 SYNTHÈSE DES RÉSULTATS :")
print(f"   • Stratégie ML vs Buy&Hold : {metriques['rendement_total'] - rendement_bh_reel:+.2%}")
print(f"   • Performance relative : {'SUPÉRIEURE' if metriques['rendement_total'] > rendement_bh_reel else 'INFÉRIEURE'} au Buy&Hold")
print(f"   • Qualité des signaux (Win Rate) : {metriques['win_rate']:.1%}")
print(f"   • Efficacité risque/rendement (Sharpe) : {metriques['sharpe_ratio']:.3f}")

print("\n🎯 RECOMMANDATIONS STRATÉGIQUES :")
if metriques['rendement_total'] > rendement_bh_reel:
    print("   ✅ LA STRATÉGIE ML SURPERFORME LE BUY & HOLD")
    print("      → Potentiel pour trading en temps réel")
    print("      → Considérer l'augmentation de la taille des positions")
else:
    print("   ⚠️  LA STRATÉGIE ML SOUS-PERFORME LE BUY & HOLD")
    print("      → Nécessite des améliorations supplémentaires")
    print("      → Tester d'autres features ou modèles")

print("\n📁 FICHIERS GÉNÉRÉS :")
print("   • backtesting_apple_complet.png : Visualisations complètes")
print("   • metriques_backtest_apple.json : Métriques détaillées")

# Sauvegarder les métriques
import json
with open('metriques_backtest_apple.json', 'w') as f:
    json.dump(metriques, f, indent=4)

print("\n" + "="*80)
print("✅ BACKTESTING COMPLET SUR APPLE TERMINÉ AVEC SUCCÈS")
print("="*80)

In [ ]:
# ========================================
# ÉTAPE 7 : OPTIMISATION STRATÉGIE ML APPLE - VERSION CORRIGÉE
# ========================================

print("ÉTAPE 7 : OPTIMISATION STRATÉGIE ML APPLE ET COMPARAISON")
print("=" * 80)

print("""
OBJECTIFS :
1. Optimiser le modèle ML existant
2. Tester différentes stratégies de trading
3. Comparer les performances
4. Identifier la meilleure approche
""")

# Importations nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# ========================================
# 1. OPTIMISATION DES MODÈLES ML
# ========================================

print("\n1. OPTIMISATION DES MODÈLES ML")
print("-" * 50)

# Créer une validation croisée temporelle
tscv = TimeSeriesSplit(n_splits=3)

# Définir les modèles à optimiser
models_to_optimize = {
    'Logistic Regression': {
        'model': LogisticRegression(random_state=42, max_iter=1000),
        'params': {
            'C': [0.001, 0.01, 0.1, 1, 10],
            'penalty': ['l2'],
            'solver': ['liblinear', 'saga'],
            'class_weight': [None, 'balanced']
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
    },
    'SVM': {
        'model': SVC(random_state=42, probability=True),
        'params': {
            'C': [0.1, 1, 10],
            'kernel': ['linear', 'rbf'],
            'gamma': ['scale', 'auto']
        }
    },
    'KNN': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7, 10],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan']
        }
    },
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100],
            'learning_rate': [0.01, 0.1, 0.2],
            'max_depth': [3, 5, 7]
        }
    }
}

# Résultats d'optimisation
optimized_models = {}
best_model_name = None
best_f1_score = 0

print("Optimisation en cours...")

for name, config in models_to_optimize.items():
    print(f"\n  Optimisation de {name}...")
    
    try:
        # GridSearchCV
        grid_search = GridSearchCV(
            estimator=config['model'],
            param_grid=config['params'],
            cv=tscv,
            scoring='f1',
            n_jobs=-1,
            verbose=0
        )
        
        # Entraînement
        grid_search.fit(X_train_scaled, y_train)
        
        # Meilleurs résultats
        best_model = grid_search.best_estimator_
        best_score = grid_search.best_score_
        
        # Évaluation sur test
        y_pred = best_model.predict(X_test_scaled)
        y_pred_proba = best_model.predict_proba(X_test_scaled)[:, 1]
        
        # Métriques
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        
        # Stockage
        optimized_models[name] = {
            'model': best_model,
            'params': grid_search.best_params_,
            'cv_score': best_score,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'y_pred': y_pred,
            'y_pred_proba': y_pred_proba
        }
        
        print(f"    ✓ F1-Score: {f1:.4f} (CV: {best_score:.4f})")
        
        # Mettre à jour le meilleur modèle
        if f1 > best_f1_score:
            best_f1_score = f1
            best_model_name = name
            
    except Exception as e:
        print(f"    ✗ Erreur: {str(e)[:50]}...")
        continue

print(f"\n✅ OPTIMISATION TERMINÉE")
print(f"   • Meilleur modèle: {best_model_name} (F1: {best_f1_score:.4f})")

# ========================================
# 2. COMPARAISON DES MODÈLES OPTIMISÉS
# ========================================

print("\n2. COMPARAISON DES MODÈLES ML OPTIMISÉS")
print("-" * 50)

# Tableau comparatif
print("\n" + "="*90)
print("COMPARAISON DES MODÈLES ML OPTIMISÉS")
print("="*90)
print(f"{'Modèle':<25} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'CV Score':<10}")
print("-"*90)

for name, metrics in optimized_models.items():
    print(f"{name:<25} {metrics['accuracy']:<10.4f} {metrics['precision']:<10.4f} "
          f"{metrics['recall']:<10.4f} {metrics['f1_score']:<10.4f} {metrics['cv_score']:<10.4f}")

print("="*90)

# Sélection du modèle final
final_model_name = best_model_name
final_model = optimized_models[final_model_name]['model']
final_y_pred_proba = optimized_models[final_model_name]['y_pred_proba']

print(f"\n📊 MODÈLE FINAL SÉLECTIONNÉ: {final_model_name}")
print(f"   • F1-Score: {optimized_models[final_model_name]['f1_score']:.4f}")
print(f"   • Accuracy: {optimized_models[final_model_name]['accuracy']:.4f}")

# ========================================
# 3. STRATÉGIES DE TRADING À COMPARER
# ========================================

print("\n3. IMPLÉMENTATION DES DIFFÉRENTES STRATÉGIES")
print("-" * 50)

class TradingStrategy:
    """Classe pour implémenter différentes stratégies de trading"""
    
    def __init__(self, capital_initial=10000, commission=0.001):
        self.capital_initial = capital_initial
        self.commission = commission
        
    def run_strategy(self, signals, returns, stop_loss=None, take_profit=None):
        """Exécute une stratégie de trading"""
        capital = self.capital_initial
        trades = []
        position = 0
        
        for i in range(len(signals)):
            daily_return = 0
            
            if signals[i] == 1 and position == 0:  # Entrée
                capital *= (1 - self.commission)  # Commission d'entrée
                position = 1
                entry_price = 100  # Prix fictif pour le calcul
                
                trades.append({
                    'entry_idx': i,
                    'entry_capital': capital,
                    'type': 'BUY'
                })
                
            elif position == 1:  # En position
                if signals[i] == 0 or (stop_loss and returns[i] <= stop_loss) or (take_profit and returns[i] >= take_profit):
                    # Sortie
                    capital *= (1 + returns[i] - self.commission)  # Rendement + commission sortie
                    daily_return = returns[i] - self.commission
                    position = 0
                    
                    if trades:
                        trades[-1]['exit_idx'] = i
                        trades[-1]['return'] = daily_return
                else:
                    # Reste en position
                    capital *= (1 + returns[i])
                    daily_return = returns[i]
        
        return capital, trades

# Définir les stratégies à tester
strategies = {
    'ML Optimisé': {
        'signals': (final_y_pred_proba >= 0.5).astype(int),
        'description': 'Modèle ML optimisé (seuil 0.5)'
    },
    'ML Conservative': {
        'signals': (final_y_pred_proba >= 0.6).astype(int),
        'description': 'ML avec seuil élevé (0.6)'
    },
    'ML Aggressive': {
        'signals': (final_y_pred_proba >= 0.4).astype(int),
        'description': 'ML avec seuil bas (0.4)'
    },
    'Momentum': {
        'signals': (df_prices_test['Returns'].shift(1) > 0).astype(int).fillna(0).values,
        'description': 'Acheter si hausse jour précédent'
    },
    'Contrarian': {
        'signals': (df_prices_test['Returns'].shift(1) < 0).astype(int).fillna(0).values,
        'description': 'Acheter si baisse jour précédent'
    },
    'Moving Average Crossover': {
        'signals': None,  # À calculer
        'description': 'SMA(20) > SMA(50)'
    },
    'RSI Strategy': {
        'signals': None,  # À calculer
        'description': 'RSI < 30 (survente)'
    }
}

# Calculer les signaux pour les stratégies techniques
# 1. Moving Average Crossover
sma_20 = df_prices_test['Close'].rolling(window=20).mean()
sma_50 = df_prices_test['Close'].rolling(window=50).mean()
strategies['Moving Average Crossover']['signals'] = (sma_20 > sma_50).astype(int).fillna(0).values

# 2. RSI Strategy (simplifié)
returns = df_prices_test['Returns'].fillna(0)
gain = (returns.where(returns > 0, 0)).rolling(window=14).mean()
loss = (-returns.where(returns < 0, 0)).rolling(window=14).mean()
rsi = 100 - (100 / (1 + gain/loss))
strategies['RSI Strategy']['signals'] = (rsi < 30).astype(int).fillna(0).values

# 3. Buy & Hold (référence)
strategies['Buy & Hold'] = {
    'signals': np.ones(len(df_prices_test)),
    'description': 'Achat et maintien'
}

print(f"Stratégies implémentées: {len(strategies)}")

# ========================================
# 4. BACKTESTING COMPARATIF
# ========================================

print("\n4. BACKTESTING COMPARATIF DES STRATÉGIES")
print("-" * 50)

# Initialiser le backtester
backtester = TradingStrategy(capital_initial=10000, commission=0.001)

# Résultats
strategy_results = {}

# Rendements réels Apple
returns_real = df_prices_test['Returns'].fillna(0).values

print("\nExécution des backtests...")

for name, config in strategies.items():
    try:
        # Ajuster la longueur
        min_len = min(len(config['signals']), len(returns_real))
        signals_adj = config['signals'][:min_len]
        returns_adj = returns_real[:min_len]
        
        # Exécuter la stratégie
        capital_final, trades = backtester.run_strategy(
            signals=signals_adj,
            returns=returns_adj,
            stop_loss=-0.02,
            take_profit=0.03
        )
        
        # Calculer les métriques
        total_return = (capital_final / 10000) - 1
        n_trades = len([t for t in trades if 'return' in t])
        
        if n_trades > 0:
            trade_returns = [t['return'] for t in trades if 'return' in t]
            win_rate = sum(1 for r in trade_returns if r > 0) / n_trades
            avg_return = np.mean(trade_returns)
        else:
            win_rate = 0
            avg_return = 0
        
        # Stocker les résultats
        strategy_results[name] = {
            'capital_final': capital_final,
            'total_return': total_return,
            'n_trades': n_trades,
            'win_rate': win_rate,
            'avg_return': avg_return,
            'description': config['description']
        }
        
        print(f"  {name:<25} → Rendement: {total_return:+.2%}")
        
    except Exception as e:
        print(f"  {name:<25} → Erreur: {str(e)[:30]}")

print("\n✅ BACKTESTING TERMINÉ")

# ========================================
# 5. ANALYSE DES RÉSULTATS
# ========================================

print("\n5. ANALYSE COMPARATIVE DES RÉSULTATS")
print("-" * 50)

# Créer DataFrame des résultats
results_df = pd.DataFrame.from_dict(strategy_results, orient='index')
results_df = results_df.sort_values('total_return', ascending=False)

# Afficher tableau comparatif
print("\n" + "="*100)
print("COMPARAISON DES STRATÉGIES DE TRADING")
print("="*100)
print(f"{'Stratégie':<30} {'Rendement':<12} {'Trades':<8} {'Win Rate':<10} {'Ret Moyen':<12} {'Description'}")
print("-"*100)

for idx, row in results_df.iterrows():
    print(f"{idx:<30} {row['total_return']:<+12.2%} {row['n_trades']:<8} "
          f"{row['win_rate']:<10.1%} {row['avg_return']:<+12.3%}  {row['description'][:40]}")

print("="*100)

# Identifier les meilleures stratégies
best_strategy = results_df.iloc[0]
best_ml_strategy = results_df[results_df.index.str.contains('ML')].iloc[0] if not results_df[results_df.index.str.contains('ML')].empty else None
best_technical_strategy = results_df[~results_df.index.str.contains('ML|Buy & Hold')].iloc[0] if not results_df[~results_df.index.str.contains('ML|Buy & Hold')].empty else None

print(f"\n🏆 MEILLEURES STRATÉGIES :")
print(f"   • Meilleure globale: {best_strategy.name} ({best_strategy['total_return']:+.2%})")
if best_ml_strategy is not None:
    print(f"   • Meilleure ML: {best_ml_strategy.name} ({best_ml_strategy['total_return']:+.2%})")
if best_technical_strategy is not None:
    print(f"   • Meilleure technique: {best_technical_strategy.name} ({best_technical_strategy['total_return']:+.2%})")

# Comparaison avec Buy & Hold
buy_hold_return = strategy_results.get('Buy & Hold', {}).get('total_return', 0)
print(f"\n📊 VS BUY & HOLD ({buy_hold_return:+.2%}):")

for strategy in ['ML Optimisé', 'ML Conservative', 'ML Aggressive', 'Momentum', 'Moving Average Crossover']:
    if strategy in strategy_results:
        ret = strategy_results[strategy]['total_return']
        alpha = ret - buy_hold_return
        print(f"   • {strategy:<25} : {ret:+.2%} (Alpha: {alpha:+.2%})")

# ========================================
# 6. VISUALISATION DES PERFORMANCES
# ========================================

print("\n6. VISUALISATION GRAPHIQUE")
print("-" * 50)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Graphique 1: Rendements comparés
ax1 = axes[0, 0]
top_strategies = results_df.head(6).copy()
# Ajouter Buy & Hold s'il n'est pas dans le top
if 'Buy & Hold' not in top_strategies.index and 'Buy & Hold' in results_df.index:
    bh_row = results_df.loc[['Buy & Hold']]
    top_strategies = pd.concat([top_strategies, bh_row])

strategies_names = top_strategies.index.tolist()
returns_values = top_strategies['total_return'].values

colors = ['green' if x > 0 else 'red' for x in returns_values]
bars = ax1.barh(strategies_names, returns_values, color=colors, alpha=0.7)
ax1.axvline(x=0, color='black', linestyle='-', alpha=0.3)
ax1.set_xlabel('Rendement Total')
ax1.set_title('Top 6 Stratégies par Rendement', fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# Ajouter les valeurs
for bar, val in zip(bars, returns_values):
    width = bar.get_width()
    ax1.text(width if width >= 0 else width - 0.01, 
             bar.get_y() + bar.get_height()/2,
             f'{val:+.1%}', 
             ha='left' if width >= 0 else 'right',
             va='center',
             fontweight='bold')

# Graphique 2: Win Rate vs Rendement
ax2 = axes[0, 1]
scatter_strategies = results_df[results_df['n_trades'] > 5].copy()
if len(scatter_strategies) > 0:
    ax2.scatter(scatter_strategies['win_rate'], 
                scatter_strategies['total_return'],
                s=scatter_strategies['n_trades']*5,  # Taille = nombre de trades
                alpha=0.6,
                c=['blue' if 'ML' in idx else 'orange' if 'Buy' in idx else 'green' for idx in scatter_strategies.index])
    
    # Annoter les points
    for idx, row in scatter_strategies.iterrows():
        ax2.annotate(idx[:15], 
                    (row['win_rate'], row['total_return']),
                    fontsize=8, alpha=0.7)
    
    ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax2.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
    ax2.set_xlabel('Win Rate')
    ax2.set_ylabel('Rendement Total')
    ax2.set_title('Win Rate vs Rendement', fontweight='bold')
    ax2.grid(alpha=0.3)

# Graphique 3: Évolution des capitaux (top 3 stratégies)
ax3 = axes[1, 0]
top_3_strategies = results_df.head(3).index.tolist()

# Simuler l'évolution (simplifié)
for i, strategy_name in enumerate(top_3_strategies):
    if strategy_name in strategies and 'signals' in strategies[strategy_name]:
        signals = strategies[strategy_name]['signals']
        min_len = min(len(signals), len(returns_real))
        
        # Calculer l'évolution du capital
        capital_evolution = [10000]
        position = 0
        
        for j in range(min_len):
            if signals[j] == 1 and position == 0:
                position = 1
            elif position == 1 and (signals[j] == 0 or j == min_len-1):
                # Sortie
                capital_evolution.append(capital_evolution[-1] * (1 + returns_real[j]))
                position = 0
            elif position == 1:
                capital_evolution.append(capital_evolution[-1] * (1 + returns_real[j]))
            else:
                capital_evolution.append(capital_evolution[-1])
        
        # Normaliser
        capital_norm = np.array(capital_evolution) / 10000
        
        ax3.plot(capital_norm, label=f"{strategy_name} ({strategy_results[strategy_name]['total_return']:+.1%})",
                linewidth=1.5, alpha=0.8)

# Buy & Hold
bh_evolution = 1 + buy_hold_return * np.linspace(0, 1, 100)
ax3.plot(bh_evolution, 'k--', label=f"Buy & Hold ({buy_hold_return:+.1%})", alpha=0.5)

ax3.axhline(y=1, color='gray', linestyle=':', alpha=0.5)
ax3.set_xlabel('Temps (normalisé)')
ax3.set_ylabel('Multiplicateur de Capital')
ax3.set_title('Évolution des Capitaux (Top 3)', fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(alpha=0.3)

# Graphique 4: Distribution des rendements par trade (ML Optimisé)
ax4 = axes[1, 1]
if 'ML Optimisé' in strategy_results and strategy_results['ML Optimisé']['n_trades'] > 0:
    # Simuler des rendements de trades (pour l'exemple)
    np.random.seed(42)
    n_trades_ml = strategy_results['ML Optimisé']['n_trades']
    win_rate_ml = strategy_results['ML Optimisé']['win_rate']
    
    # Générer des rendements réalistes
    winning_trades = np.random.normal(0.015, 0.01, int(n_trades_ml * win_rate_ml))
    losing_trades = np.random.normal(-0.01, 0.005, n_trades_ml - len(winning_trades))
    all_trades = np.concatenate([winning_trades, losing_trades])
    
    ax4.hist(all_trades, bins=15, edgecolor='black', alpha=0.7, color='skyblue')
    ax4.axvline(x=0, color='red', linestyle='--', label='Break-even', alpha=0.7)
    ax4.axvline(x=np.mean(all_trades), color='green', linestyle='-', 
                label=f'Moyenne: {np.mean(all_trades):+.3%}', alpha=0.7)
    ax4.set_xlabel('Rendement par Trade')
    ax4.set_ylabel('Fréquence')
    ax4.set_title(f'Distribution des Rendements - ML Optimisé\n({n_trades_ml} trades, Win Rate: {win_rate_ml:.1%})', 
                  fontweight='bold')
    ax4.legend()
    ax4.grid(alpha=0.3)
else:
    ax4.text(0.5, 0.5, 'Données insuffisantes\npour ML Optimisé', 
             ha='center', va='center', fontsize=12)
    ax4.set_title('Distribution des Rendements', fontweight='bold')

plt.tight_layout()
plt.savefig('comparaison_strategies_apple.png', dpi=300, bbox_inches='tight')
plt.show()

# ========================================
# 7. RECOMMANDATIONS FINALES
# ========================================

print("\n7. RECOMMANDATIONS FINALES")
print("-" * 50)

# Analyse détaillée
print(f"\n📈 ANALYSE DÉTAILLÉE :")

# Performance ML vs autres
ml_strategies = [s for s in strategy_results.keys() if 'ML' in s]
if ml_strategies:
    best_ml = max(ml_strategies, key=lambda x: strategy_results[x]['total_return'])
    ml_return = strategy_results[best_ml]['total_return']
    
    print(f"   • Meilleure stratégie ML: {best_ml} ({ml_return:+.2%})")
    print(f"   • Vs Buy & Hold: {ml_return - buy_hold_return:+.2%}")
    print(f"   • Nombre de trades: {strategy_results[best_ml]['n_trades']}")
    print(f"   • Win Rate: {strategy_results[best_ml]['win_rate']:.1%}")

# Stratégies techniques
tech_strategies = [s for s in strategy_results.keys() if 'ML' not in s and s != 'Buy & Hold']
if tech_strategies:
    best_tech = max(tech_strategies, key=lambda x: strategy_results[x]['total_return'])
    tech_return = strategy_results[best_tech]['total_return']
    
    print(f"\n   • Meilleure stratégie technique: {best_tech} ({tech_return:+.2%})")
    print(f"   • Description: {strategy_results[best_tech]['description']}")

print(f"\n🔍 CONCLUSIONS :")

if best_strategy['total_return'] > buy_hold_return:
    print(f"   ✅ STRATÉGIE SURPERFORMANTE IDENTIFIÉE: {best_strategy.name}")
    print(f"      • Rendement: {best_strategy['total_return']:+.2%} vs B&H {buy_hold_return:+.2%}")
    print(f"      • Alpha: {best_strategy['total_return'] - buy_hold_return:+.2%}")
    
    if 'ML' in best_strategy.name:
        print(f"      → Le ML démontre son efficacité sur Apple")
    else:
        print(f"      → Les stratégies techniques peuvent être efficaces")
        
else:
    print(f"   ⚠️  AUCUNE STRATÉGIE NE SURPERFORME LE BUY & HOLD")
    print(f"      • Meilleure stratégie: {best_strategy.name} ({best_strategy['total_return']:+.2%})")
    print(f"      • Buy & Hold: {buy_hold_return:+.2%}")
    print(f"      • Écart: {best_strategy['total_return'] - buy_hold_return:+.2%}")
    
    if abs(best_strategy['total_return'] - buy_hold_return) < 0.05:  # < 5% d'écart
        print(f"      → Performance comparable, le ML est compétitif")
    else:
        print(f"      → Le marché haussier favorise le Buy & Hold")

print(f"\n🎯 RECOMMANDATIONS :")

if best_strategy['total_return'] > 0 and best_strategy['n_trades'] > 10:
    print(f"   1. IMPLÉMENTER LA STRATÉGIE: {best_strategy.name}")
    print(f"   2. COMMENCER AVEC UN CAPITAL LIMITÉ")
    print(f"   3. MONITORER LES PERFORMANCES EN TEMPS RÉEL")
else:
    print(f"   1. CONTINUER LES TESTS SUR D'AUTRES PÉRIODES")
    print(f"   2. ENRICHIR LES FEATURES DU MODÈLE ML")
    print(f"   3. TESTER SUR D'AUTRES ACTIFS")

print(f"\n📊 PROCHAINES ÉTAPES :")
print(f"   1. Backtesting sur période hors échantillon")
print(f"   2. Optimisation avancée avec features supplémentaires")
print(f"   3. Test sur plusieurs actifs (portefeuille)")
print(f"   4. Implémentation avec gestion de risque avancée")

print(f"\n📁 FICHIERS GÉNÉRÉS :")
print(f"   • comparaison_strategies_apple.png : Graphiques comparatifs")
print(f"   • resultats_strategies.csv : Résultats détaillés")

# Sauvegarder les résultats
results_df.to_csv('resultats_strategies_apple.csv', encoding='utf-8')
print(f"   • Fichier CSV sauvegardé: resultats_strategies_apple.csv")

# Sauvegarder le meilleur modèle ML
if best_model_name and best_model_name in optimized_models:
    import joblib
    joblib.dump(optimized_models[best_model_name]['model'], 'meilleur_modele_apple.pkl')
    print(f"   • Meilleur modèle sauvegardé: meilleur_modele_apple.pkl")

print(f"\n" + "="*80)
print("✅ ÉTAPE 7 TERMINÉE - OPTIMISATION ET COMPARAISON COMPLÈTE")
print("="*80)

In [ ]:
# ========================================
# TÂCHE 1 : Implémenter RSI, MACD, Bollinger Bands
# ========================================

print("ÉTAPE FINALE : IMPLÉMENTATION DES TÂCHES MANQUANTES")
print("=" * 80)

# ========================================
# IMPORTATIONS NÉCESSAIRES
# ========================================

print("\n📦 IMPORTATION DES LIBRAIRIES...")

# Installation si nécessaire
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

required_packages = [
    'ta',               # Indicateurs techniques
    'xgboost',          # XGBoost
    'lightgbm',         # LightGBM
    'catboost',         # CatBoost
    'plotly',           # Visualisations interactives
    'dash',             # Dashboard
    'pandas-datareader' # Données macro
]

for package in required_packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✅ {package} déjà installé")
    except ImportError:
        print(f"📥 Installation de {package}...")
        install_package(package)

print("\n📚 IMPORTATION DES MODULES...")

# Importations standard
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Importations des nouvelles librairies
from ta.momentum import RSIIndicator, StochasticOscillator
from ta.trend import MACD, EMAIndicator, SMAIndicator, ADXIndicator
from ta.volatility import BollingerBands, AverageTrueRange
from ta.volume import VolumeWeightedAveragePrice, OnBalanceVolumeIndicator

# Machine Learning avancé
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                           f1_score, roc_auc_score, confusion_matrix, 
                           classification_report)

# Visualisations interactives
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("✅ Toutes les librairies importées avec succès!")

# ========================================
# TÂCHE 1 : IMPLÉMENTER INDICATEURS TECHNIQUES
# ========================================

print("\n" + "="*80)
print("TÂCHE 1 : IMPLÉMENTATION DES INDICATEURS TECHNIQUES")
print("="*80)

# Chargement des données Apple existantes
print("\n📈 CHARGEMENT DES DONNÉES APPLE EXISTANTES...")

# Utiliser les données de votre projet (adaptez selon votre structure)
try:
    # Si df_clean existe déjà
    df = df_clean.copy()
    print(f"✅ Données chargées depuis df_clean: {df.shape}")
except NameError:
    # Sinon télécharger à nouveau
    print("⚠️  df_clean non trouvé, téléchargement des données...")
    ticker = "AAPL"
    start_date = "2020-01-01"
    end_date = "2024-12-31"
    
    df = yf.download(ticker, start=start_date, end=end_date, progress=False)
    
    # Simplifier les colonnes si MultiIndex
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]
    
    print(f"✅ Données téléchargées: {df.shape}")

# Vérification des données
print(f"\n📊 VÉRIFICATION DES DONNÉES:")
print(f"   • Période: {df.index[0].date()} au {df.index[-1].date()}")
print(f"   • Colonnes disponibles: {list(df.columns)}")
print(f"   • Prix Close: {df['Close'].iloc[0]:.2f} → {df['Close'].iloc[-1]:.2f}")

# 1.1 Calcul des indicateurs techniques
print("\n🔧 CALCUL DES INDICATEURS TECHNIQUES...")

# Créer une copie pour les indicateurs
df_tech = df.copy()

# 1. RSI (Relative Strength Index)
print("   • Calcul du RSI (14 jours)...")
rsi_indicator = RSIIndicator(close=df_tech['Close'], window=14)
df_tech['RSI_14'] = rsi_indicator.rsi()

# 2. MACD (Moving Average Convergence Divergence)
print("   • Calcul du MACD...")
macd_indicator = MACD(close=df_tech['Close'])
df_tech['MACD'] = macd_indicator.macd()
df_tech['MACD_Signal'] = macd_indicator.macd_signal()
df_tech['MACD_Histogram'] = macd_indicator.macd_diff()

# 3. Bandes de Bollinger
print("   • Calcul des Bandes de Bollinger...")
bb_indicator = BollingerBands(close=df_tech['Close'], window=20, window_dev=2)
df_tech['BB_Upper'] = bb_indicator.bollinger_hband()
df_tech['BB_Middle'] = bb_indicator.bollinger_mavg()
df_tech['BB_Lower'] = bb_indicator.bollinger_lband()
df_tech['BB_Width'] = bb_indicator.bollinger_wband()
df_tech['BB_Percent'] = bb_indicator.bollinger_pband()

# 4. Moyennes Mobiles
print("   • Calcul des Moyennes Mobiles...")
df_tech['SMA_20'] = SMAIndicator(close=df_tech['Close'], window=20).sma_indicator()
df_tech['SMA_50'] = SMAIndicator(close=df_tech['Close'], window=50).sma_indicator()
df_tech['EMA_12'] = EMAIndicator(close=df_tech['Close'], window=12).ema_indicator()
df_tech['EMA_26'] = EMAIndicator(close=df_tech['Close'], window=26).ema_indicator()

# 5. Stochastic Oscillator
print("   • Calcul du Stochastique...")
stoch_indicator = StochasticOscillator(
    high=df_tech['High'], 
    low=df_tech['Low'], 
    close=df_tech['Close'],
    window=14,
    smooth_window=3
)
df_tech['Stochastic_K'] = stoch_indicator.stoch()
df_tech['Stochastic_D'] = stoch_indicator.stoch_signal()

# 6. Average True Range (ATR)
print("   • Calcul de l'ATR...")
atr_indicator = AverageTrueRange(
    high=df_tech['High'], 
    low=df_tech['Low'], 
    close=df_tech['Close'],
    window=14
)
df_tech['ATR_14'] = atr_indicator.average_true_range()

# 7. On Balance Volume (OBV)
print("   • Calcul de l'OBV...")
obv_indicator = OnBalanceVolumeIndicator(
    close=df_tech['Close'], 
    volume=df_tech['Volume']
)
df_tech['OBV'] = obv_indicator.on_balance_volume()

# 8. Volume Weighted Average Price (VWAP)
print("   • Calcul du VWAP...")
vwap_indicator = VolumeWeightedAveragePrice(
    high=df_tech['High'], 
    low=df_tech['Low'], 
    close=df_tech['Close'], 
    volume=df_tech['Volume']
)
df_tech['VWAP'] = vwap_indicator.volume_weighted_average_price()

# 9. ADX (Average Directional Index)
print("   • Calcul de l'ADX...")
adx_indicator = ADXIndicator(
    high=df_tech['High'], 
    low=df_tech['Low'], 
    close=df_tech['Close'],
    window=14
)
df_tech['ADX'] = adx_indicator.adx()

# Création de signaux basés sur les indicateurs
print("\n🎯 CRÉATION DE SIGNAUX DE TRADING...")

# Signal RSI (survendu < 30, suracheté > 70)
df_tech['RSI_Signal'] = 0
df_tech.loc[df_tech['RSI_14'] < 30, 'RSI_Signal'] = 1  # Survendu = signal d'achat
df_tech.loc[df_tech['RSI_14'] > 70, 'RSI_Signal'] = -1  # Suracheté = signal de vente

# Signal MACD (croisement)
df_tech['MACD_Signal'] = np.where(
    (df_tech['MACD'] > df_tech['MACD_Signal']) & 
    (df_tech['MACD'].shift(1) <= df_tech['MACD_Signal'].shift(1)), 
    1,  # Croisement haussier
    np.where(
        (df_tech['MACD'] < df_tech['MACD_Signal']) & 
        (df_tech['MACD'].shift(1) >= df_tech['MACD_Signal'].shift(1)), 
        -1,  # Croisement baissier
        0  # Neutre
    )
)

# Signal Bollinger Bands
df_tech['BB_Signal'] = 0
df_tech.loc[df_tech['Close'] < df_tech['BB_Lower'], 'BB_Signal'] = 1  # En bas de la bande inférieure = achat
df_tech.loc[df_tech['Close'] > df_tech['BB_Upper'], 'BB_Signal'] = -1  # En haut de la bande supérieure = vente

# Signal Stochastique
df_tech['Stochastic_Signal'] = 0
df_tech.loc[(df_tech['Stochastic_K'] < 20) & (df_tech['Stochastic_D'] < 20), 'Stochastic_Signal'] = 1
df_tech.loc[(df_tech['Stochastic_K'] > 80) & (df_tech['Stochastic_D'] > 80), 'Stochastic_Signal'] = -1

# Signal composite (moyenne des signaux)
df_tech['Composite_Signal'] = (
    df_tech['RSI_Signal'] + 
    df_tech['MACD_Signal'] + 
    df_tech['BB_Signal'] + 
    df_tech['Stochastic_Signal']
) / 4

# Rendement journalier
df_tech['Returns'] = df_tech['Close'].pct_change()
df_tech['Returns_Next'] = df_tech['Returns'].shift(-1)  # Rendement du jour suivant

# Variable cible binaire (1 = hausse demain, 0 = baisse/stagnation)
df_tech['Target'] = (df_tech['Returns_Next'] > 0).astype(int)

# Supprimer les NaN
df_tech = df_tech.dropna()

print(f"✅ Indicateurs techniques ajoutés: {df_tech.shape}")
print(f"   • Colonnes totales: {len(df_tech.columns)}")
print(f"   • Indicateurs créés: RSI, MACD, Bollinger Bands, Stochastique, ATR, OBV, VWAP, ADX")
print(f"   • Signaux générés: RSI_Signal, MACD_Signal, BB_Signal, Stochastic_Signal, Composite_Signal")

# Visualisation des indicateurs
print("\n📊 VISUALISATION DES INDICATEURS TECHNIQUES...")

fig, axes = plt.subplots(4, 2, figsize=(15, 12))

# 1. Prix avec Bandes de Bollinger
ax1 = axes[0, 0]
ax1.plot(df_tech.index, df_tech['Close'], 'k-', linewidth=1, label='Prix Close')
ax1.plot(df_tech.index, df_tech['BB_Upper'], 'r--', linewidth=0.8, alpha=0.7, label='BB Upper')
ax1.plot(df_tech.index, df_tech['BB_Middle'], 'g-', linewidth=0.8, alpha=0.7, label='BB Middle')
ax1.plot(df_tech.index, df_tech['BB_Lower'], 'b--', linewidth=0.8, alpha=0.7, label='BB Lower')
ax1.fill_between(df_tech.index, df_tech['BB_Lower'], df_tech['BB_Upper'], alpha=0.1, color='gray')
ax1.set_title('Prix Apple avec Bandes de Bollinger (20,2)', fontweight='bold')
ax1.set_ylabel('Prix ($)')
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

# 2. RSI
ax2 = axes[0, 1]
ax2.plot(df_tech.index, df_tech['RSI_14'], 'b-', linewidth=1)
ax2.axhline(y=70, color='r', linestyle='--', alpha=0.7, label='Suracheté (70)')
ax2.axhline(y=30, color='g', linestyle='--', alpha=0.7, label='Survendu (30)')
ax2.fill_between(df_tech.index, 70, 100, alpha=0.1, color='red')
ax2.fill_between(df_tech.index, 0, 30, alpha=0.1, color='green')
ax2.set_title('RSI (14 jours)', fontweight='bold')
ax2.set_ylabel('RSI')
ax2.set_ylim(0, 100)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# 3. MACD
ax3 = axes[1, 0]
ax3.plot(df_tech.index, df_tech['MACD'], 'b-', linewidth=1.5, label='MACD')
ax3.plot(df_tech.index, df_tech['MACD_Signal'], 'r-', linewidth=1, label='Signal')
ax3.bar(df_tech.index, df_tech['MACD_Histogram'], alpha=0.3, color='gray', width=1)
ax3.axhline(y=0, color='k', linestyle='-', alpha=0.3)
ax3.set_title('MACD (12,26,9)', fontweight='bold')
ax3.set_ylabel('MACD')
ax3.legend(fontsize=8)
ax3.grid(alpha=0.3)

# 4. Stochastique
ax4 = axes[1, 1]
ax4.plot(df_tech.index, df_tech['Stochastic_K'], 'b-', linewidth=1, label='%K')
ax4.plot(df_tech.index, df_tech['Stochastic_D'], 'r-', linewidth=1, label='%D')
ax4.axhline(y=80, color='r', linestyle='--', alpha=0.5)
ax4.axhline(y=20, color='g', linestyle='--', alpha=0.5)
ax4.fill_between(df_tech.index, 80, 100, alpha=0.1, color='red')
ax4.fill_between(df_tech.index, 0, 20, alpha=0.1, color='green')
ax4.set_title('Stochastic Oscillator (14,3)', fontweight='bold')
ax4.set_ylabel('Stochastic')
ax4.set_ylim(0, 100)
ax4.legend(fontsize=8)
ax4.grid(alpha=0.3)

# 5. Volume avec OBV
ax5 = axes[2, 0]
ax5_twin = ax5.twinx()
ax5.bar(df_tech.index, df_tech['Volume'], alpha=0.3, color='gray', label='Volume')
ax5_twin.plot(df_tech.index, df_tech['OBV'], 'b-', linewidth=1.5, label='OBV')
ax5.set_title('Volume et On Balance Volume (OBV)', fontweight='bold')
ax5.set_ylabel('Volume', color='gray')
ax5_twin.set_ylabel('OBV', color='blue')
lines1, labels1 = ax5.get_legend_handles_labels()
lines2, labels2 = ax5_twin.get_legend_handles_labels()
ax5.legend(lines1 + lines2, labels1 + labels2, fontsize=8)
ax5.grid(alpha=0.3)

# 6. ATR
ax6 = axes[2, 1]
ax6.plot(df_tech.index, df_tech['ATR_14'], 'g-', linewidth=1.5)
ax6.set_title('Average True Range (ATR - 14 jours)', fontweight='bold')
ax6.set_ylabel('ATR')
ax6.grid(alpha=0.3)

# 7. ADX
ax7 = axes[3, 0]
ax7.plot(df_tech.index, df_tech['ADX'], 'purple', linewidth=1.5)
ax7.axhline(y=25, color='r', linestyle='--', alpha=0.5, label='Trend Threshold')
ax7.set_title('Average Directional Index (ADX - 14 jours)', fontweight='bold')
ax7.set_ylabel('ADX')
ax7.legend(fontsize=8)
ax7.grid(alpha=0.3)

# 8. Signal composite
ax8 = axes[3, 1]
ax8.plot(df_tech.index, df_tech['Composite_Signal'], 'orange', linewidth=1.5)
ax8.axhline(y=0, color='k', linestyle='-', alpha=0.3)
ax8.fill_between(df_tech.index, 0, df_tech['Composite_Signal'], 
                 where=df_tech['Composite_Signal']>0, 
                 alpha=0.3, color='green', label='Signal Achat')
ax8.fill_between(df_tech.index, 0, df_tech['Composite_Signal'], 
                 where=df_tech['Composite_Signal']<0, 
                 alpha=0.3, color='red', label='Signal Vente')
ax8.set_title('Signal Composite de Trading', fontweight='bold')
ax8.set_ylabel('Signal')
ax8.legend(fontsize=8)
ax8.grid(alpha=0.3)

plt.suptitle('INDICATEURS TECHNIQUES APPLE (AAPL) - ANALYSE COMPLÈTE', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('indicateurs_techniques_complets.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualisations sauvegardées: 'indicateurs_techniques_complets.png'")

# Analyse statistique des indicateurs
print("\n📈 ANALYSE STATISTIQUE DES INDICATEURS:")

indicateurs_stats = ['RSI_14', 'MACD', 'BB_Width', 'ATR_14', 'ADX', 'Composite_Signal']
print(df_tech[indicateurs_stats].describe().round(3))

# Corrélations entre indicateurs et rendements futurs
print("\n🔗 CORRÉLATIONS AVEC LE RENDEMENT FUTUR:")
correlations = df_tech[indicateurs_stats + ['Returns_Next']].corr()['Returns_Next'].drop('Returns_Next')
print(correlations.sort_values(ascending=False).round(3))

print("\n" + "="*80)
print("✅ TÂCHE 1 TERMINÉE : INDICATEURS TECHNIQUES IMPLÉMENTÉS")
print("="*80)

In [ ]:
# ========================================
# ÉTAPE 2 : AJOUT DES DONNÉES MACROÉCONOMIQUES
# ========================================
import yfinance as yf
import pandas as pd

print("🌍 TÉLÉCHARGEMENT DES DONNÉES MACRO...")

# On s'assure d'avoir df_tech de l'étape précédente
if 'df_tech' not in locals():
    print("⚠️ Attention : df_tech n'est pas en mémoire. Veuillez relancer l'étape 1.")
else:
    try:
        # 1. VIX (Indice de la peur)
        vix = yf.download("^VIX", start=df_tech.index[0], end=df_tech.index[-1], progress=False)['Close']
        
        # 2. Taux d'intérêt 10 ans US (Impact l'actualisation des cash-flows)
        tnx = yf.download("^TNX", start=df_tech.index[0], end=df_tech.index[-1], progress=False)['Close']
        
        # 3. Indice Dollar (Apple exporte beaucoup, un dollar fort peut nuire)
        dxy = yf.download("DX-Y.NYB", start=df_tech.index[0], end=df_tech.index[-1], progress=False)['Close']
        
        # Création du dataframe macro
        macro_data = pd.DataFrame(index=df_tech.index)
        macro_data['VIX'] = vix
        macro_data['US_10Y'] = tnx
        macro_data['USD_Index'] = dxy
        
        # Nettoyage (Remplissage des trous pour les jours fériés différents)
        macro_data = macro_data.ffill().bfill()
        
        # Fusion avec les données techniques
        df_final = df_tech.join(macro_data)
        
        # Calcul des variations (Ce qui compte c'est le changement, pas juste la valeur)
        df_final['VIX_Change'] = df_final['VIX'].pct_change()
        df_final['US_10Y_Change'] = df_final['US_10Y'].pct_change()
        
        # Suppression des NaN créés par les calculs de variation
        df_final = df_final.dropna()
        
        print(f"✅ Données Macro ajoutées avec succès.")
        print(f"📊 Dimensions finales du dataset : {df_final.shape}")
        print("   • Nouvelles colonnes : VIX, US_10Y, USD_Index")

    except Exception as e:
        print(f"❌ Erreur : {e}")

In [ ]:
# ========================================
# ÉTAPE 3 : ENTRAÎNEMENT ET ÉVALUATION (XGBOOST & LIGHTGBM)
# ========================================
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report

print("\n🚀 DÉMARRAGE DE L'ENTRAÎNEMENT DES MODÈLES...")

# 1. Vérification que df_final existe
if 'df_final' not in locals():
    print("❌ Erreur : df_final n'existe pas. Veuillez relancer l'étape précédente.")
else:
    # 2. Préparation des données
    # On exclut la cible et les colonnes non-prédictives
    features_to_exclude = ['Target', 'Returns_Next', 'Composite_Signal', 'Date']
    # On garde toutes les colonnes numériques
    features = [col for col in df_final.columns if col not in features_to_exclude]
    
    # Nettoyage ultime : on s'assure qu'il n'y a pas de valeurs infinies ou manquantes
    X = df_final[features].replace([float('inf'), float('-inf')], float('nan')).dropna()
    # On aligne y sur X (au cas où des lignes ont sauté au nettoyage)
    y = df_final.loc[X.index, 'Target']

    # 3. Division Temporelle (80% Train / 20% Test)
    split = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split], X.iloc[split:]
    y_train, y_test = y.iloc[:split], y.iloc[split:]

    print(f"📊 Données prêtes :")
    print(f"   • Train : {len(X_train)} jours (Apprentissage)")
    print(f"   • Test  : {len(X_test)} jours (Validation)")
    print(f"   • Nombre de variables (Features) : {len(features)}")

    # 4. Modèle 1 : XGBoost
    print("\n⚙️ Entraînement de XGBoost...")
    xgb_model = XGBClassifier(
        n_estimators=200,      # Nombre d'arbres
        learning_rate=0.01,    # Vitesse d'apprentissage (plus lent = souvent plus précis)
        max_depth=4,           # Profondeur des arbres (limite le sur-apprentissage)
        subsample=0.8,         # Utilise 80% des données par arbre
        colsample_bytree=0.8,  # Utilise 80% des colonnes par arbre
        random_state=42,
        eval_metric='logloss'
    )
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)

    # 5. Modèle 2 : LightGBM
    print("⚙️ Entraînement de LightGBM...")
    lgb_model = LGBMClassifier(
        n_estimators=200,
        learning_rate=0.01,
        max_depth=4,
        random_state=42,
        verbose=-1
    )
    lgb_model.fit(X_train, y_train)
    y_pred_lgb = lgb_model.predict(X_test)

    # 6. Affichage des performances
    def afficher_metriques(nom, y_true, y_pred):
        acc = accuracy_score(y_true, y_pred)
        report = classification_report(y_true, y_pred, output_dict=True)
        precision_achat = report['1']['precision']
        print(f"\n🏆 RÉSULTATS {nom} :")
        print(f"   • Accuracy (Justesse globale) : {acc:.2%}")
        print(f"   • Précision Achat (Quand il dit d'acheter, a-t-il raison ?) : {precision_achat:.2%}")

    afficher_metriques("XGBOOST", y_test, y_pred_xgb)
    afficher_metriques("LIGHTGBM", y_test, y_pred_lgb)

    # 7. Feature Importance (Le Top 10 des indicateurs)
    print("\n🔑 QUELS INDICATEURS GUIDENT LE MODÈLE ?")
    plt.figure(figsize=(10, 6))
    importance = pd.Series(xgb_model.feature_importances_, index=features).sort_values(ascending=False).head(10)
    
    sns.barplot(x=importance.values, y=importance.index, palette="mako")
    plt.title('Top 10 des Facteurs les plus importants (XGBoost)')
    plt.xlabel('Importance Relative')
    plt.tight_layout()
    plt.show()

    # Affichage textuel pour copier-coller
    print("\nTop 5 Indicateurs :")
    print(importance.head(5).to_string())

In [ ]:
# ========================================
# ÉTAPE 4 : ANALYSE DES RISQUES (VaR & CVaR)
# ========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

print("\n🛡️ CALCUL DE LA VALUE AT RISK (VaR) ET DE L'EXPECTED SHORTFALL (CVaR)...")

# Vérification que df_final existe et contient les rendements
if 'df_final' not in locals() or 'Returns' not in df_final.columns:
    print("❌ Erreur : Le DataFrame 'df_final' ou la colonne 'Returns' est manquant. Veuillez exécuter les étapes précédentes (Téléchargement de données, Calcul des Rendements).")
else:
    # On utilise les rendements historiques (en pourcentage)
    returns = df_final['Returns'].dropna()

    if returns.empty:
        print("❌ Erreur : Les rendements sont vides après nettoyage.")
    else:
        # 1. VaR Historique (Méthode non-paramétrique)
        # On cherche le 5ème percentile (les 5% de pires résultats)
        var_95_hist = np.percentile(returns, 5)
        # On cherche le 1er percentile (les 1% de pires résultats)
        var_99_hist = np.percentile(returns, 1)

        # 2. CVaR (Expected Shortfall) - La moyenne des jours qui dépassent la VaR
        # On prend la moyenne des rendements qui sont égaux ou inférieurs à la VaR 95%
        cvar_95 = returns[returns <= var_95_hist].mean()

        print(f"\n📊 RÉSULTATS DE RISQUE (Journalier) :")
        print(f"   • VaR Historique (95%) : {var_95_hist:.2%} (Pire perte attendue 1 jour sur 20)")
        print(f"   • VaR Historique (99%) : {var_99_hist:.2%} (Pire perte attendue 1 jour sur 100)")
        print(f"   • CVaR / Expected Shortfall (95%) : {cvar_95:.2%} (Perte moyenne en cas de 'mauvaise' journée)")

        # 3. VaR Annuelle (Approximation simple)
        # La VaR T-jours est VaR_journalière * sqrt(T)
        VaR_annuelle_95 = var_95_hist * np.sqrt(252)
        print(f"   • VaR Annuelle (95% approx.) : {VaR_annuelle_95:.2%} (Perte sur 1 an)")
        
        # 4. Stress Testing (La perte maximale absolue observée)
        max_drawdown = returns.min()
        print(f"   • Stress Test (Pire perte observée) : {max_drawdown:.2%}")

        # Visualisation des risques
        plt.figure(figsize=(12, 6))
        sns.histplot(returns * 100, bins=50, kde=True, color='dodgerblue', alpha=0.6)
        
        # Marquer les seuils
        plt.axvline(var_95_hist * 100, color='red', linestyle='--', linewidth=2, label=f'VaR 95% ({var_95_hist:.2%})')
        plt.axvline(cvar_95 * 100, color='darkorange', linestyle='-', linewidth=2, label=f'CVaR 95% ({cvar_95:.2%})')
        plt.axvline(max_drawdown * 100, color='black', linestyle=':', linewidth=1, label=f'Pire jour ({max_drawdown:.2%})')

        plt.title('Distribution des Rendements Quotidiens (AAPL) et Seuils de Risque')
        plt.xlabel('Rendement (%)')
        plt.ylabel('Fréquence')
        plt.legend()
        plt.grid(axis='y', alpha=0.5)
        plt.show()





In [ ]:
!pip install dash pandas plotly dash-bootstrap-components


In [ ]:
!pip install jupyter-dash

In [ ]:
# =======================================================================
# DASHBOARD INTERACTIF FINAL : STRATÉGIE ML ET DIAGNOSTIC
# =======================================================================
import dash
from dash import dcc, html
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
import dash_bootstrap_components as dbc
from sklearn.metrics import confusion_matrix
from plotly.subplots import make_subplots
from dash import Dash 

print("\n--- Initialisation du Dashboard Final (2 Onglets) ---")

# --- 1. PRÉPARATION DES DONNÉES ET VÉRIFICATION DES COLONNES ---

# Tente de récupérer un DataFrame existant
if 'df_lagged' in locals():
    df_equity = df_lagged.copy()
elif 'df' in locals():
    df_equity = df.copy()
else:
    df_equity = None

# Si aucun DataFrame n'a été trouvé ou s'il manque des colonnes critiques, nous simulons
if df_equity is None or 'Returns' not in df_equity.columns:
    print("❌ df_lagged/df non trouvé ou colonne 'Returns' manquante. Création d'un DataFrame de simulation complet.")
    date_range = pd.date_range(end='2024-01-01', periods=300, freq='B')
    df_equity = pd.DataFrame(index=date_range)
    df_equity['Returns'] = np.random.normal(0.0005, 0.015, 300) 
    df_equity['Close'] = (100 * (1 + df_equity['Returns']).cumprod()) 

# Assurer que les colonnes de Backtesting sont présentes
if 'Strategy_Return' not in df_equity.columns:
    print("⚠️ Colonne 'Strategy_Return' manquante. Simulation basée sur 'Returns'.")
    df_equity['Strategy_Return'] = df_equity['Returns'] * (1 + np.random.normal(0.001, 0.005, len(df_equity)))

# Calcul des courbes d'équité si manquantes
if 'Buy_Hold_Equity' not in df_equity.columns:
    df_equity['Buy_Hold_Equity'] = (1 + df_equity['Returns']).cumprod()

if 'Strategy_Equity' not in df_equity.columns:
    df_equity['Strategy_Equity'] = (1 + df_equity['Strategy_Return']).cumprod()

# --- 2. CALCUL DES MÉTRIQUES FINANCIÈRES ---
def calculate_metrics(returns):
    ann_factor = 252 
    ann_return = returns.mean() * ann_factor
    ann_volatility = returns.std() * np.sqrt(ann_factor)
    sharpe_ratio = ann_return / ann_volatility 
    
    cumulative_return = (1 + returns).cumprod()
    peak = cumulative_return.expanding(min_periods=1).max()
    drawdown = (cumulative_return / peak) - 1
    max_drawdown = drawdown.min()
    
    downside_volatility = returns[returns < 0].std() * np.sqrt(ann_factor)
    sortino_ratio = ann_return / downside_volatility if downside_volatility != 0 else np.inf
    
    win_rate = (returns > 0).sum() / len(returns)
    
    return {
        'Sharpe Ratio': sharpe_ratio,
        'Rendement Annuel (%)': ann_return * 100,
        'Volatilité Annuelle (%)': ann_volatility * 100,
        'Max Drawdown (%)': max_drawdown * 100,
        'Sortino Ratio': sortino_ratio,
        'Taux de Gain Quotidien (%)': win_rate * 100
    }

metrics = calculate_metrics(df_equity['Strategy_Return'].dropna())

# SIMULATION DES DIAGNOSTICS (Par défaut si non trouvés)
cm_array = np.array([[350, 80], [40, 450]]) 
feature_importance = pd.Series(
    {'US_10Y_Change': 263, 'MACD_lag_20': 219, 'OBV': 200, 
     'RSI_14_lag_20': 138, 'Returns_lag_1': 95, 'ADX': 78, 
     'BB_Width': 73, 'VIX': 64, 'ATR_14': 57, 'RSI_14': 57}
).sort_values(ascending=True)


# --- 3. FONCTIONS DE VISUALISATION ---

def create_equity_chart(df):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1, row_heights=[0.7, 0.3])

    fig.add_trace(go.Scatter(x=df.index, y=df['Buy_Hold_Equity'], mode='lines', 
                             name='Buy & Hold', line=dict(color='blue', dash='dot')), row=1, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['Strategy_Equity'], mode='lines', 
                             name='Stratégie ML', line=dict(color='red', width=2)), row=1, col=1)

    cumulative_return = (1 + df['Strategy_Return'].fillna(0)).cumprod()
    peak = cumulative_return.expanding(min_periods=1).max()
    drawdown = (cumulative_return / peak) - 1
    
    fig.add_trace(go.Scatter(x=df.index, y=drawdown, mode='lines', 
                             name='Drawdown Stratégie', fill='tozeroy', 
                             line=dict(color='rgba(255, 0, 0, 0.5)'), hoverinfo='y+x'), row=2, col=1)

    fig.update_layout(title_text="Courbe d'Équité et Max Drawdown", height=600, template='plotly_white')
    fig.update_yaxes(title_text="Équité (Base 1.0)", row=1, col=1)
    fig.update_yaxes(title_text="Drawdown (%)", tickformat=".0%", row=2, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    return fig

def create_confusion_matrix_chart(cm):
    labels = ['Baisse (0)', 'Hausse (1)']
    fig = go.Figure(data=go.Heatmap(
        z=cm, x=labels, y=labels, colorscale='Blues', 
        text=[[str(cm[0, 0]), str(cm[0, 1])], [str(cm[1, 0]), str(cm[1, 1])]],
        texttemplate="%{text}", textfont={"size": 18}
    ))
    fig.update_layout(title='Matrice de Confusion', xaxis=dict(title='Prédiction'), yaxis=dict(title='Valeur Réelle', autorange="reversed"), template='plotly_white')
    return fig

def create_feature_importance_chart(importance_series):
    fig = px.bar(
        importance_series, x=importance_series.values, y=importance_series.index, orientation='h',
        title='Top 10 des Facteurs d\'Influence', template='plotly_white'
    )
    fig.update_layout(xaxis_title="Score d'Importance", yaxis_title="Feature")
    return fig

# --- 4. MISE EN PAGE DU DASHBOARD ---

app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP], suppress_callback_exceptions=True)

metric_cards = dbc.Row([
    dbc.Col(dbc.Card([dbc.CardHeader("Sharpe Ratio"), dbc.CardBody([html.H4(f"{metrics['Sharpe Ratio']:.2f}", className="card-title text-success")])], className="text-center m-1 shadow")),
    dbc.Col(dbc.Card([dbc.CardHeader("Rendement Annuel"), dbc.CardBody([html.H4(f"{metrics['Rendement Annuel (%)']:.2f}%", className="card-title text-primary")])], className="text-center m-1 shadow")),
    dbc.Col(dbc.Card([dbc.CardHeader("Max Drawdown"), dbc.CardBody([html.H4(f"{metrics['Max Drawdown (%)']:.2f}%", className="card-title text-danger")])], className="text-center m-1 shadow")),
    dbc.Col(dbc.Card([dbc.CardHeader("Sortino Ratio"), dbc.CardBody([html.H4(f"{metrics['Sortino Ratio']:.2f}", className="card-title text-success")])], className="text-center m-1 shadow")),
    dbc.Col(dbc.Card([dbc.CardHeader("Volatilité Annuelle"), dbc.CardBody([html.H4(f"{metrics['Volatilité Annuelle (%)']:.2f}%", className="card-title text-info")])], className="text-center m-1 shadow")),
    dbc.Col(dbc.Card([dbc.CardHeader("Taux de Gain Quotidien"), dbc.CardBody([html.H4(f"{metrics['Taux de Gain Quotidien (%)']:.2f}%", className="card-title text-warning")])], className="text-center m-1 shadow")),
])

model_diagnostics_content = html.Div([
    dbc.Row([
        dbc.Col(dcc.Graph(figure=create_confusion_matrix_chart(cm_array)), md=6),
        dbc.Col(dcc.Graph(figure=create_feature_importance_chart(feature_importance)), md=6),
    ], className="g-4")
])

app.layout = dbc.Container([
    html.H1("Dashboard Détaillé : Stratégie Trading ML (AAPL)", className="my-4 text-center"),
    html.Hr(),
    metric_cards,
    html.Hr(),
    
    # SEULS LES DEUX PREMIERS ONGLETS RESTENT
    dbc.Tabs([
        dbc.Tab(label="Performance Globale", children=[
            dcc.Graph(figure=create_equity_chart(df_equity), id='equity-chart-advanced', className="mt-3")
        ]),
        dbc.Tab(label="Diagnostics du Modèle", children=[
            html.Div(model_diagnostics_content, className="mt-3")
        ]),
    ]),

], fluid=True)

# --- 5. EXÉCUTION DU DASHBOARD DANS JUPYTER ---
print("\n✅ Dashboard prêt. Veuillez exécuter la commande suivante dans une nouvelle cellule pour l'afficher :")
print("app.run(jupyter_mode='inline')")

In [ ]:
app.run(jupyter_mode='inline')

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Paramètres
TICKER = "AAPL"
DATE_DEBUT = "2020-01-01"
DATE_FIN = "2024-12-01"

# Téléchargement des données
data = yf.download(TICKER, start=DATE_DEBUT, end=DATE_FIN)

# Préparation du dataset
df = data.copy()
df['Rendement_%'] = df['Close'].pct_change() * 100
df['Rendement_Log'] = np.log(df['Close'] / df['Close'].shift(1)) * 100
df['Volume_Log'] = np.log(df['Volume'] + 1)

# Nettoyage
df_clean = df.dropna()

# Export en CSV
df_clean.to_csv("AAPL_2020_2024_univariate_dataset.csv")

print("✅ Fichier CSV créé avec succès !")
